# SFT Dataset Generation — from `passed_mcqs.jsonl` (S5 Output)

**AI Tutor System — Teacher-Student Knowledge Distillation**

---

## Purpose

This notebook builds the final **Supervised Fine-Tuning (SFT) dataset** for Qwen student-model training.  
It reads from the **S5 validation pipeline output** (`passed_mcqs.jsonl`) — the
highest-quality, fully-validated MCQs that survived all five validation stages — and
converts each slot-level record into a **LLaMA-Factory–compatible SFT training sample**.

---

## Input → Output

| | File | Format | Contents |
|---|---|---|---|
| **Input** | `passed_mcqs.jsonl` | JSONL | S5 stage-by-stage `passed.jsonl` records (one per validated MCQ slot) |
| **Input** | `selected_subset_v3.jsonl` | JSONL | S3 chunks (provides `metadata`, `keywords`, `context_fringe`) |
| **Output** | `sft_metadata_conditioned.jsonl` | JSONL | Metadata-conditioned SFT records (Option A) |
| **Output** | `sft_dataset_info.json` | JSON | LLaMA-Factory `dataset_info.json` entries |

---

## Input Record Schema (`passed_mcqs.jsonl`)

Each line in `passed_mcqs.jsonl` is a S5 stage-by-stage `passed.jsonl` record:

```json
{
  "mcq_data": {
    "chunk_id":   "<str>",
    "text":       "<str>  — the source chunk text",
    "slot_index": "<int>",
    "chunk_type": "<str>  — definition | process | comparison | example | argument",
    "concept":    "<str>  — the concept this slot tests",
    "bloom_level":"<str>  — remember | understand | apply | analyze | evaluate | create",
    "mcq": {
      "question":    "<str>",
      "options":     {"A": "<str>", "B": "<str>", "C": "<str>", "D": "<str>"},
      "answer":      "<A|B|C|D>",
      "explanation": "<str>"
    }
  },
  "chunk_id":    "<str>",
  "slot_index":  "<int>",
  "stage_name":  "llm_judge",
  "stage_number": 5,
  "timestamp":   "<ISO-8601 str>",
  "validation_result": {
    "passed": true,
    "reason": "",
    "scores": { "correctness": 4.0, "clarity": 4.0, "distractors": 3.5,
                "educational_value": 4.0, "chunk_grounding": 4.0, "overall": 3.9 },
    "ambiguous": false
  },
  "scores": {}
}
```

---

## SFT Format: `sharegpt` (LLaMA-Factory)

Every training record uses the **`sharegpt`** conversation format with three turns:

```
system   → static domain expert persona + output format rules
user     → chunk text + metadata conditioning tags + task instruction
assistant → the validated MCQ as a clean JSON object
```

This matches the **runtime inference format exactly** — no train/inference mismatch.

---

## Two Training Variants

| Variant | User turn includes metadata? | Purpose |
|---|---|---|
| **Option A** `sft_metadata_conditioned.jsonl` | Yes — `[BLOOM LEVEL]`, `[CHUNK TYPE]`, `[FOCUS CONCEPT]` | Primary training target; enables runtime Bloom control |


## 1. Environment Setup

> **Kaggle Environment** — This notebook runs on **Kaggle** (not Colab, no Drive).
>
> **Input files** (read-only Kaggle Dataset):
> ```
> /kaggle/input/datasets/ayakhaled51/mcqs-files/
>   ├── passed_mcqs.jsonl
>   └── selected_subset_v3.jsonl
> ```
>
> **All new outputs** go to `/kaggle/working/` (writable, persists for the session):
> ```
> /kaggle/working/
>   ├── GP_Quiz-Generation_Data/      ← generated SFT files, split files
>   ├── LLaMA-Factory/                ← cloned repo
>   ├── mcq-lora-adapter/             ← training output (pushed to HF, then cleaned)
>   └── mcq-qwen-merged/              ← merged model (pushed to HF, then cleaned)
> ```


In [2]:
# ─── Kaggle Environment Setup ─────────────────────────────────────────────────
# No Google Drive. Input files come from the Kaggle Dataset mounted at:
#   /kaggle/input/datasets/ayakhaled51/mcqs-files/
# All generated files go to /kaggle/working/

import os

# ── Input paths (read-only Kaggle Dataset) ────────────────────────────────────
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/ayakhaled51/mcqs-files"
# ── Output root (writable) ────────────────────────────────────────────────────
KAGGLE_WORKING   = "/kaggle/working"
OUTPUT_DATA_DIR  = os.path.join(KAGGLE_WORKING, "GP_Quiz-Generation_Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

# ── Convenience alias used throughout the notebook ───────────────────────────
# INPUT_DIR  → read from (Kaggle Dataset, read-only)
# OUTPUT_DIR → write to  (/kaggle/working, writable)
INPUT_DIR  = KAGGLE_INPUT_DIR
OUTPUT_DIR = OUTPUT_DATA_DIR

# ── Load HF token from Kaggle Secrets ────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _secrets   = UserSecretsClient()
    HF_TOKEN   = _secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✓ HF_TOKEN loaded from Kaggle Secrets.")
except Exception as _e:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print(f"  HF_TOKEN fallback to env var: {'found' if HF_TOKEN else 'MISSING'}  ({_e})")

# ── Verify input dataset is accessible ───────────────────────────────────────
print()
print(f"Input  dir : {INPUT_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print()
if os.path.exists(INPUT_DIR):
    files = os.listdir(INPUT_DIR)
    print(f"✓ Input dataset found — {len(files)} file(s):")
    for fn in sorted(files):
        size_mb = os.path.getsize(os.path.join(INPUT_DIR, fn)) / (1024**2)
        print(f"    {fn}  ({size_mb:.1f} MB)")
else:
    print(f"✗ Input dataset NOT found at {INPUT_DIR}")
    print("  → Make sure you added the dataset 'ayakhaled51/input-data' to this notebook")
    print("    via Notebook Settings → Add Data.")

✓ HF_TOKEN loaded from Kaggle Secrets.

Input  dir : /kaggle/input/datasets/ayakhaled51/mcqs-files
Output dir : /kaggle/working/GP_Quiz-Generation_Data

✓ Input dataset found — 7 file(s):
    Qwen2.5-1.5B-Instruct  (0.0 MB)
    passed_mcqs.jsonl  (6.5 MB)
    selected_subset_v3.jsonl  (1.6 MB)
    selected_subset_v4.jsonl  (0.9 MB)
    sft_dataset_info.json  (0.0 MB)
    sft_metadata_conditioned.jsonl  (13.4 MB)
    teacher_outputs.jsonl  (0.2 MB)


In [79]:
# ─── Base Model Download ──────────────────────────────────────────────────────
# Downloads Qwen2.5-1.5B-Instruct to /kaggle/working/base_models/ on first run.
# Subsequent runs skip the download if the folder already exists.

from transformers import AutoModelForCausalLM, AutoTokenizer
import os

base_id       = "Qwen/Qwen2.5-1.5B-Instruct"
base_save_dir = os.path.join(KAGGLE_WORKING, "base_models/Qwen2.5-1.5B-Instruct")

if not os.path.exists(base_save_dir):
    print(f"Downloading base model → {base_save_dir} ...")
    tokenizer = AutoTokenizer.from_pretrained(base_id, token=HF_TOKEN)
    model     = AutoModelForCausalLM.from_pretrained(base_id, token=HF_TOKEN)
    model.save_pretrained(base_save_dir)
    tokenizer.save_pretrained(base_save_dir)
    del model
    import gc; gc.collect()
    print("✓ Base model saved.")
else:
    print(f"✓ Base model already exists: {base_save_dir}")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Base model saved.


In [4]:
import os

MODEL_PATH = "/kaggle/input/datasets/ayakhaled51/mcqs-files/Qwen2.5-1.5B-Instruct/Qwen2.5-1.5B-Instruct"

# Check files
print(os.listdir(MODEL_PATH))

print("✓ Model path loaded successfully")

['config.json', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'model-001.safetensors', 'generation_config.json']
✓ Model path loaded successfully


In [5]:
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM

SRC_DIR = "/kaggle/input/datasets/ayakhaled51/mcqs-files/Qwen2.5-1.5B-Instruct/Qwen2.5-1.5B-Instruct"
WORK_DIR = "/kaggle/working/qwen_model"

os.makedirs(WORK_DIR, exist_ok=True)

# copy all files from input to working
for name in os.listdir(SRC_DIR):
    src_path = os.path.join(SRC_DIR, name)
    dst_path = os.path.join(WORK_DIR, name)
    if os.path.isfile(src_path):
        shutil.copy2(src_path, dst_path)

# rename the safetensors file to the name transformers expects
old_file = os.path.join(WORK_DIR, "model-001.safetensors")
new_file = os.path.join(WORK_DIR, "model.safetensors")

if os.path.exists(old_file) and not os.path.exists(new_file):
    os.rename(old_file, new_file)

print("Files in WORK_DIR:", os.listdir(WORK_DIR))

tokenizer = AutoTokenizer.from_pretrained(WORK_DIR, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    WORK_DIR,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype="auto"
)

print("✓ Model loaded")

Files in WORK_DIR: ['model.safetensors', 'config.json', 'tokenizer_config.json', 'generation_config.json', 'chat_template.jinja', 'tokenizer.json']


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✓ Model loaded


In [6]:
import json
import os
import random
from collections import Counter
from os.path import join
from typing import Any, Dict, List, Optional, Tuple

print("Imports OK")

Imports OK


## 2. Paths & Configuration

In [7]:
# ─── Paths & Configuration ────────────────────────────────────────────────────
import os
from os.path import join

# ── Read from Kaggle Dataset (read-only) ─────────────────────────────────────
S5_PASSED_PATH = join(INPUT_DIR, "passed_mcqs.jsonl")
S3_CHUNKS_PATH = join(INPUT_DIR, "selected_subset_v3.jsonl")

# ── Write to /kaggle/working (writable) ───────────────────────────────────────
SFT_META_PATH  = join(OUTPUT_DIR, "sft_metadata_conditioned.jsonl")
SFT_INFO_PATH  = join(OUTPUT_DIR, "sft_dataset_info.json")

# ── Validate inputs exist ─────────────────────────────────────────────────────
for path in [S5_PASSED_PATH, S3_CHUNKS_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Required input not found: {path}\n"
            f"Make sure 'ayakhaled51/mcqs-files' is added as a dataset to this notebook."
        )

print(f"S5 passed MCQs  : {S5_PASSED_PATH}")
print(f"S3 chunks       : {S3_CHUNKS_PATH}")
print(f"SFT output      : {SFT_META_PATH}")
print(f"Dataset info    : {SFT_INFO_PATH}")

S5 passed MCQs  : /kaggle/input/datasets/ayakhaled51/mcqs-files/passed_mcqs.jsonl
S3 chunks       : /kaggle/input/datasets/ayakhaled51/mcqs-files/selected_subset_v3.jsonl
SFT output      : /kaggle/working/GP_Quiz-Generation_Data/sft_metadata_conditioned.jsonl
Dataset info    : /kaggle/working/GP_Quiz-Generation_Data/sft_dataset_info.json


## 3. Load Inputs

### 3.1  Load S5 Passed MCQs

Each record is a S5 stage-by-stage `passed.jsonl` entry.  
The validated MCQ and its conditioning metadata live inside `mcq_data`.

In [8]:
def load_jsonl(path: str) -> List[Dict]:
    """Load a JSONL file, skipping blank lines and malformed records."""
    records: List[Dict] = []
    skipped = 0
    with open(path, encoding="utf-8") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"  [WARN] Skipping malformed line {lineno}: {exc}")
                skipped += 1
    if skipped:
        print(f"  Skipped {skipped} malformed line(s) in {os.path.basename(path)}")
    return records


s5_records: List[Dict] = load_jsonl(S5_PASSED_PATH)
print(f"Loaded {len(s5_records):,} passed MCQ records from S5.")

Loaded 2,921 passed MCQ records from S5.


### 3.2  Load S3 Chunks

S3 chunks provide the **keywords** and **context_fringe** fields that were stored
in the original chunk but are not carried through the S5 slot-level records.

If a chunk_id from S5 is not found in S3, the SFT record is still built using
only the data available in `mcq_data` (graceful degradation — keywords default
to empty, context fringe defaults to absent).

In [9]:
# Build lookup: chunk_id → full S3 chunk dict
s3_lookup: Dict[str, Dict] = {}
for rec in load_jsonl(S3_CHUNKS_PATH):
    cid = rec.get("chunk_id")
    if cid:
        s3_lookup[cid] = rec

print(f"S3 chunk lookup : {len(s3_lookup):,} chunks")

# Coverage check
s5_chunk_ids  = {r.get("chunk_id", r.get("mcq_data", {}).get("chunk_id", ""))
                 for r in s5_records}
matched       = s5_chunk_ids & set(s3_lookup.keys())
unmatched     = s5_chunk_ids - set(s3_lookup.keys())

print(f"Unique chunk_ids in S5 : {len(s5_chunk_ids):,}")
print(f"Matched in S3          : {len(matched):,}")
if unmatched:
    print(f"NOT matched (keywords/fringe will be empty): {len(unmatched):,}")
    for cid in list(unmatched)[:5]:
        print(f"  {cid}")

S3 chunk lookup : 945 chunks
Unique chunk_ids in S5 : 937
Matched in S3          : 937


## 4. Record Extraction Helper

`extract_slot_data()` normalises a single S5 passed record into a clean,
flat dict containing only the fields needed for SFT construction.

**Fields kept:**

| Field | Source | Purpose |
|---|---|---|
| `chunk_id` | `mcq_data` | Record identity / dedup key |
| `slot_index` | `mcq_data` | Slot identity within the chunk |
| `text` | `mcq_data` | Source chunk text → user turn |
| `bloom_level` | `mcq_data` | Bloom conditioning tag |
| `chunk_type` | `mcq_data` | Chunk-type conditioning tag |
| `concept` | `mcq_data` | Focus concept conditioning tag |
| `keywords` | S3 lookup | Distractor-quality hint in user turn |
| `prev_sentence` | S3 lookup | Context fringe (background only) |
| `next_sentence` | S3 lookup | Context fringe (background only) |
| `mcq` | `mcq_data` | The validated MCQ → assistant turn |

**Fields excluded:**

| Field | Reason for exclusion |
|---|---|
| `file_id` | Internal pipeline routing; irrelevant to model training |
| `s5_validation` | Validation bookkeeping; quality is implicit in the curated dataset itself |
| `chunk_type_confidence` | Classifier confidence; not meaningful in a training context |
| `stage_name`, `stage_number` | Pipeline provenance; not a training signal |
| `timestamp` | Pipeline provenance; not a training signal |
| `scores` (top-level) | Redundant empty dict from S5 per-MCQ records |
| `validation_result` | Pipeline bookkeeping; removed entirely from SFT output |

In [10]:
def extract_slot_data(
    s5_record: Dict,
    s3_lookup: Dict[str, Dict],
) -> Optional[Dict]:
    """
    Extract and normalise training-relevant fields from one S5 passed record.

    Parameters
    ----------
    s5_record : one line from passed_mcqs.jsonl
    s3_lookup : chunk_id -> S3 chunk dict (for keywords + context_fringe)

    Returns
    -------
    Flat dict with exactly the fields needed for SFT construction,
    or None if the record is structurally invalid.
    """
    # ── 1. Pull mcq_data sub-dict ─────────────────────────────────────────────
    mcq_data: Dict = s5_record.get("mcq_data", {})
    if not mcq_data:
        return None

    # ── 2. Required fields from mcq_data ─────────────────────────────────────
    chunk_id  : str       = mcq_data.get("chunk_id", "").strip()
    slot_index: int       = mcq_data.get("slot_index", 0)
    text      : str       = mcq_data.get("text", "").strip()
    bloom     : str       = mcq_data.get("bloom_level", "understand").strip().lower()
    chunk_type: str       = mcq_data.get("chunk_type", "definition").strip().lower()
    concept   : str       = mcq_data.get("concept", "").strip()
    mcq       : Dict      = mcq_data.get("mcq", {})

    # Validate the minimum required fields
    if not chunk_id or not text or not mcq:
        return None

    # Validate MCQ sub-structure
    if not mcq.get("question") or not mcq.get("options") or not mcq.get("answer"):
        return None

    if set(mcq.get("options", {}).keys()) != {"A", "B", "C", "D"}:
        return None

    # ── 3. Enrich from S3 lookup (keywords + context fringe) ─────────────────
    s3_chunk    : Dict        = s3_lookup.get(chunk_id, {})
    s3_meta     : Dict        = s3_chunk.get("metadata", {})
    fringe      : Dict        = s3_chunk.get("context_fringe", {})

    keywords    : List[str]   = s3_meta.get("keywords", [])
    prev_sentence: str        = (fringe.get("prev_sentence") or "").strip()
    next_sentence: str        = (fringe.get("next_sentence") or "").strip()

    # ── 4. Build clean MCQ dict (only the 4 training-relevant fields) ─────────
    clean_mcq = {
        "question":    mcq.get("question", "").strip(),
        "options":     mcq.get("options", {}),
        "answer":      mcq.get("answer", "").strip().upper(),
        "explanation": mcq.get("explanation", "").strip(),
    }

    # ── 5. Return flat, clean dict ────────────────────────────────────────────
    return {
        "chunk_id":     chunk_id,
        "slot_index":   slot_index,
        "text":         text,
        "bloom_level":  bloom,
        "chunk_type":   chunk_type,
        "concept":      concept,
        "keywords":     keywords,
        "prev_sentence": prev_sentence,
        "next_sentence": next_sentence,
        "mcq":          clean_mcq,
    }


print("extract_slot_data() defined.")

extract_slot_data() defined.


## 5. Prompts — Preserved from S4 Generation Pipeline

The system prompt and user prompt structure are **preserved exactly** from the
original S4 generation pipeline so the student model's training format matches
its inference format precisely.

### Changes from the original `build_sft_record()`

| What changed | Why |
|---|---|
| Input source is `passed_mcqs.jsonl` (S5), not `s4_success.jsonl` | Uses only the highest-quality, fully-validated MCQs |
| One SFT record per **slot** (not per chunk) | Training signal is per MCQ, not per chunk |
| `sharegpt` conversation format with `system`/`user`/`assistant` turns | Direct LLaMA-Factory compatibility; matches runtime inference format |
| Assistant turn is a **clean JSON object** (not wrapped in a JSON array) | One slot = one MCQ = one JSON object; no array needed |
| Metadata tags use natural-language bracket format `[BLOOM LEVEL]` | Better adherence in instruction-tuned models than raw JSON metadata |
| `
| `file_id`, `s5_validation`, `chunk_type_confidence` fully removed | Pipeline metadata; zero training value |


In [11]:
# ── System Prompt ─────────────────────────────────────────────────────────────
# Preserved from the S4 generation pipeline (SYSTEM_PROMPT).
# Kept as a static string — identical across all training records.
# NOT repeated in the user turn (matches how S4 sent it as system_instruction).

SFT_SYSTEM_PROMPT = "\n".join([
    # -- Persona & core behavior
    "You are an expert educational content designer for Bloom's Taxonomy-aligned assessment.",
    "Generate multiple-choice questions from educational text.",
    "Return ONLY valid JSON. No markdown fences, preambles, or commentary.",
    "",

    # -- Bloom levels (compressed)
    "## BLOOM LEVELS",
    "remember: recall facts | understand: explain meaning | apply: use in scenario",
    "analyze: compare/reason | evaluate: justify decision | create: design/propose",
    "",

    # -- Chunk type forms (compressed)
    "## CHUNK TYPE FORMS",
    "definition: explain concept | process: steps/sequence | comparison: differences",
    "example: real case | argument: evaluate claim",
    "",

    # -- Grounding rule (highest priority)
    "## GROUNDING (highest priority)",
    "- Correct answer MUST be explicitly stated in or directly inferable from the chunk.",
    "- Do NOT use external knowledge.",
    "",

    # -- MCQ quality rules
    "## MCQ RULES",
    "- Exactly 4 options: A, B, C, D. Exactly ONE correct answer.",
    "- Forbidden: 'All of the above', 'None of the above', 'Both A and B'.",
    "- Options must be similar in length (±25-30%) and share the same grammatical structure.",
    "- Options must be mutually exclusive. Do NOT embed the correct answer text in the question.",
    "",

    # -- Distractor rules
    "## DISTRACTORS",
    "- Use the keywords list to craft plausible distractors.",
    "- All 3 wrong options must belong to the same category/domain as the correct answer.",
    "- Distractors must be plausible to a student who skipped the chunk.",
    "",

    # -- Question phrasing
    "## QUESTION PHRASING",
    '- Do NOT say "according to the text", "based on the chunk", or "as stated above".',
    "- Question stem must be self-contained and <= 50 words.",
    "",

    "- You MUST complete the JSON fully. Do not stop mid-output.",
    "- Ensure all 4 options A,B,C,D are present before finishing.",
    "",

    "## EXPLANATION RULES",
    "- Do NOT mention option letters (A, B, C, D).",
    "- Refer to the answer by its content, not its position.",
    "- Do NOT use phrases like 'according to the text' or 'according to the chunk'.",
    "- Justify the correct concept clearly using its meaning.",
    "",

    # -- Output format (single MCQ object)
    "## OUTPUT FORMAT — return ONLY this JSON object:",
    "{",
    '  "question": "<full question stem>",',
    '  "options": {"A": "...", "B": "...", "C": "...", "D": "..."},',
    '  "answer": "<A|B|C|D>",',
    '  "explanation": "<1-2 sentences: why correct, referencing chunk>"',
    "}",
])

print(f"SFT_SYSTEM_PROMPT  : {len(SFT_SYSTEM_PROMPT):,} chars  (~{len(SFT_SYSTEM_PROMPT)//4} tokens)")

SFT_SYSTEM_PROMPT  : 2,024 chars  (~506 tokens)


In [12]:
# ── Bloom and Chunk-Type instruction dicts ───────────────────────────────────
# Preserved from the S4 pipeline (BLOOM_INSTRUCTION, CHUNK_TYPE_FORM).
# Used to build the

BLOOM_INSTRUCTION: Dict[str, str] = {
    "remember"  : "Generate a RECALL question. Test a fact, definition, or named item directly stated in the text.",
    "understand": "Generate an INTERPRETATION question. Test whether the student can explain or paraphrase.",
    "apply"     : "Generate a SCENARIO question. Place the concept in a new, concrete situation.",
    "analyze"   : "Generate a COMPARISON or REASONING question. The stem MUST compare two items or ask for a cause.",
    "evaluate"  : "Generate a JUDGMENT question. Ask the student to choose and justify.",
    "create"    : "Generate a DESIGN or SYNTHESIS question.",
}

CHUNK_TYPE_FORM: Dict[str, str] = {
    "definition": "Use 'What is / Which best describes' style.",
    "process"   : "Use 'What happens when / Which step follows' style.",
    "comparison": "Use 'How does X differ from Y / Which distinguishes' style.",
    "example"   : "Use a scenario-based style embedding the concept in a concrete case.",
    "argument"  : "Require evaluating a claim, premise, or conclusion.",
}

print("BLOOM_INSTRUCTION and CHUNK_TYPE_FORM defined.")

BLOOM_INSTRUCTION and CHUNK_TYPE_FORM defined.


In [13]:
# ── User Prompt Builders ──────────────────────────────────────────────────────

def build_user_prompt_conditioned(slot: Dict) -> str:
    """
    Build the user turn for OPTION A (metadata-conditioned) SFT records.

    Structure (preserved from S4 build_user_prompt, extended with structured tags):

      ## CHUNK TEXT          — the source text
      ## CONTEXT             — context fringe (prev/next sentences, background only)
      ## METADATA            — chunk_type + keywords  (from S4)
      ---
      [BLOOM LEVEL]          — explicit conditioning tag
      [CHUNK TYPE]           — explicit conditioning tag
      [FOCUS CONCEPT]        — explicit conditioning tag
      ---
      ## YOUR TASK           — task instruction

    Parameters
    ----------
    slot : clean dict from extract_slot_data()
    """
    text        = slot["text"]
    bloom       = slot["bloom_level"]
    chunk_type  = slot["chunk_type"]
    concept     = slot["concept"] or "the main concept in this chunk"
    keywords    = slot["keywords"]
    prev_s      = slot["prev_sentence"]
    next_s      = slot["next_sentence"]

    bloom_instr = BLOOM_INSTRUCTION.get(bloom, BLOOM_INSTRUCTION["understand"])
    ct_instr    = CHUNK_TYPE_FORM.get(chunk_type,
                      "Use a general question style appropriate to the content.")
    kw_str      = ", ".join(keywords) if keywords else "(infer from text)"

    # Context fringe block (only included if present)
    context_lines = []
    if prev_s:
        context_lines.append(f"Previous: {prev_s}")
    if next_s:
        context_lines.append(f"Next    : {next_s}")

    parts = [
        "## CHUNK TEXT",
        text,
        "",
    ]

    if context_lines:
        parts += [
            "---",
            "",
            "## CONTEXT (background only — do NOT write questions about these sentences)",
        ] + context_lines + [""]

    parts += [
        "---",
        "",
        "## METADATA",
        f"- chunk_type : {chunk_type}",
        f"- keywords   : {kw_str}",
        "",
        "---",
        "",
        # ── Structured conditioning tags ──────────────────────────────────────
        f"[BLOOM LEVEL]: {bloom}",
        f"[CHUNK TYPE]: {chunk_type}",
        f"[FOCUS CONCEPT]: {concept}",
        "",
        "CHUNK TYPE INSTRUCTION",
        ct_instr,
        "",
        "---",
        "",
        "## YOUR TASK FOR THIS QUESTION",
        "Generate EXACTLY ONE MCQ. Not two. Not zero. One.",
        "Return a single JSON OBJECT (not an array).",
        f'The question MUST focus specifically on: "{concept}"',
        "The correct answer MUST be grounded in the chunk text above.",
    ]

    return "\n".join(parts)


print("User prompt builders defined: build_user_prompt_conditioned")

User prompt builders defined: build_user_prompt_conditioned


## 6. SFT Record Builders

`build_sft_record_conditioned()` and `build_sft_record_direct()` produce
LLaMA-Factory `sharegpt`-format records:

```json
{
  "id": "<unique record id>",
  "conversations": [
    {"role": "system",    "content": "<static system prompt>"},
    {"role": "user",      "content": "<chunk + metadata tags + task>"},
    {"role": "assistant", "content": "<validated MCQ as clean JSON string>"}
  ]
}
```

The assistant turn is a **clean JSON string** (not wrapped in markdown fences,
not an array) because:
- The system prompt explicitly instructs: *Return ONLY valid JSON. No markdown fences.*
- The student model must learn to produce raw JSON for its outputs to be parsed
  without post-processing at inference time.

In [14]:
def _mcq_to_assistant_content(mcq: Dict) -> str:
    """
    Serialise the validated MCQ dict as a clean JSON string.

    - No markdown fences  (system prompt forbids them)
    - No array wrapper    (one slot = one MCQ object)
    - Compact but readable (indent=2 for human inspection; LLaMA-Factory ignores it)
    - Only the 4 MCQ fields: question, options, answer, explanation
    """
    return json.dumps(
        {
            "question":    mcq["question"],
            "options":     mcq["options"],
            "answer":      mcq["answer"],
            "explanation": mcq["explanation"],
        },
        ensure_ascii=False,
        indent=2,
    )


def build_sft_record_conditioned(slot: Dict) -> Dict:
    """
    Build a LLaMA-Factory sharegpt SFT record for OPTION A
    (metadata-conditioned generation).

    Record schema
    -------------
    {
      "id":            "<chunk_id>_slot<slot_index>_meta",
      "conversations": [
        {"role": "system",    "content": SFT_SYSTEM_PROMPT},
        {"role": "user",      "content": <conditioned user prompt>},
        {"role": "assistant", "content": <MCQ JSON string>}
      ]
    }

    Excluded fields: file_id, s5_validation, chunk_type_confidence,
                     stage_name, stage_number, timestamp, validation_result
    """
    return {
        "id": f"{slot['chunk_id']}_slot{slot['slot_index']}_meta",
        "conversations": [
            {
                "role":    "system",
                "content": SFT_SYSTEM_PROMPT,
            },
            {
                "role":    "user",
                "content": build_user_prompt_conditioned(slot),
            },
            {
                "role":    "assistant",
                "content": _mcq_to_assistant_content(slot["mcq"]),
            },
        ],
    }


print("SFT record builders defined: build_sft_record_conditioned")

SFT record builders defined: build_sft_record_conditioned


## 7. Build SFT Dataset

Processes every record in `passed_mcqs.jsonl`:

1. Extract and validate slot data via `extract_slot_data()`
2. Deduplicate by `(chunk_id, slot_index)` — keeps the first occurrence
3. Build Option A records
4. Write both output files

In [15]:
def build_sft_dataset(
    s5_records : List[Dict],
    s3_lookup  : Dict[str, Dict],
    meta_path  : str,
) -> Dict[str, int]:
    """
    Convert all S5 passed records into SFT training samples.

    Writes the Option A output file.

    Returns a stats dict.
    """

    stats = {
        "total_s5_records" : len(s5_records),
        "extracted"        : 0,
        "skipped_invalid"  : 0,
        "skipped_duplicate": 0,
        "written"          : 0,
    }

    seen_keys: set = set()   # (chunk_id, slot_index) — dedup guard

    with open(meta_path, "w", encoding="utf-8") as meta_f:

        # ── LOOP OVER ALL S5 RECORDS ───────────────────────────────
        for s5_rec in s5_records:

            # ── 1. Extract clean slot data ────────────────────────
            slot = extract_slot_data(s5_rec, s3_lookup)

            if slot is None:
                stats["skipped_invalid"] += 1
                continue

            stats["extracted"] += 1

            # ── 2. Deduplicate ───────────────────────────────────
            key = (slot["chunk_id"], slot["slot_index"])

            if key in seen_keys:
                stats["skipped_duplicate"] += 1
                continue

            seen_keys.add(key)

            # ── 3. Build SFT record ──────────────────────────────
            meta_rec = build_sft_record_conditioned(slot)

            meta_f.write(
                json.dumps(meta_rec, ensure_ascii=False) + "\n"
            )

            stats["written"] += 1

    return stats


# ── Run ───────────────────────────────────────────────────────────
stats = build_sft_dataset(
    s5_records = s5_records,
    s3_lookup  = s3_lookup,
    meta_path  = SFT_META_PATH
)

print("\n── SFT Dataset Build Complete ──────────────────────────")
print(f"  Total S5 records    : {stats['total_s5_records']:>6,}")
print(f"  Extracted (valid)   : {stats['extracted']:>6,}")
print(f"  Skipped (invalid)   : {stats['skipped_invalid']:>6,}")
print(f"  Skipped (duplicate) : {stats['skipped_duplicate']:>6,}")
print(f"  Written             : {stats['written']:>6,}")

print(f"\n  Output → {SFT_META_PATH}")


── SFT Dataset Build Complete ──────────────────────────
  Total S5 records    :  2,921
  Extracted (valid)   :  2,921
  Skipped (invalid)   :      0
  Skipped (duplicate) :      0
  Written             :  2,921

  Output → /kaggle/working/GP_Quiz-Generation_Data/sft_metadata_conditioned.jsonl


## 8. Generate `dataset_info.json` for LLaMA-Factory

LLaMA-Factory requires a `dataset_info.json` entry for each dataset file.
This cell generates the entries and saves them to disk.

In [16]:
dataset_info = {

    # ── Option A: metadata-conditioned (primary training set) ───────────────
    "mcq_sft_conditioned": {
        "file_name": os.path.basename(SFT_META_PATH),

        "formatting": "sharegpt",

        "columns": {
            "messages": "conversations"
        },

        "tags": {
            "role_tag"      : "role",
            "content_tag"   : "content",
            "user_tag"      : "user",
            "assistant_tag" : "assistant",
            "system_tag"    : "system"
        }
    }
}

with open(SFT_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"dataset_info.json written → {SFT_INFO_PATH}")
print()

print("Copy these entries into LLaMA-Factory's data/dataset_info.json:")
print(json.dumps(dataset_info, indent=2))

dataset_info.json written → /kaggle/working/GP_Quiz-Generation_Data/sft_dataset_info.json

Copy these entries into LLaMA-Factory's data/dataset_info.json:
{
  "mcq_sft_conditioned": {
    "file_name": "sft_metadata_conditioned.jsonl",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations"
    },
    "tags": {
      "role_tag": "role",
      "content_tag": "content",
      "user_tag": "user",
      "assistant_tag": "assistant",
      "system_tag": "system"
    }
  }
}


## 9. Inspection & Quality Checks

In [17]:
# ── Reload and compute distribution statistics ────────────────────────────────
sft_records: List[Dict] = load_jsonl(SFT_META_PATH)

# Recover per-slot metadata from the clean id field
# (chunk_id and slot_index are embedded in the id, but we need to re-parse
#  slots from S5 for the stats — so we use the already-built slot list)

# Rebuild slot list for stats (does NOT re-write files)
slots_for_stats: List[Dict] = []
seen_stat_keys: set = set()
for s5_rec in s5_records:
    slot = extract_slot_data(s5_rec, s3_lookup)
    if slot is None:
        continue
    key = (slot["chunk_id"], slot["slot_index"])
    if key in seen_stat_keys:
        continue
    seen_stat_keys.add(key)
    slots_for_stats.append(slot)

bloom_dist  = Counter(s["bloom_level"] for s in slots_for_stats)
type_dist   = Counter(s["chunk_type"]  for s in slots_for_stats)
answer_dist = Counter(s["mcq"]["answer"] for s in slots_for_stats)
has_kw      = sum(1 for s in slots_for_stats if s["keywords"])
has_fringe  = sum(1 for s in slots_for_stats if s["prev_sentence"] or s["next_sentence"])

print(f"Total SFT records   : {len(sft_records):,}")
print()
print("── Bloom Level Distribution ────────────────────────────")
for level in ["remember", "understand", "apply", "analyze", "evaluate", "create"]:
    count = bloom_dist.get(level, 0)
    pct   = count / max(1, len(slots_for_stats)) * 100
    bar   = "█" * int(pct / 2)
    print(f"  {level:<12}: {count:>5}  ({pct:4.1f}%)  {bar}")

print()
print("── Chunk Type Distribution ─────────────────────────────")
for ctype in ["definition", "process", "comparison", "example", "argument"]:
    count = type_dist.get(ctype, 0)
    pct   = count / max(1, len(slots_for_stats)) * 100
    print(f"  {ctype:<12}: {count:>5}  ({pct:4.1f}%)")

print()
print("── Answer Key Distribution (should be ~25% each) ───────")
for key in ["A", "B", "C", "D"]:
    count = answer_dist.get(key, 0)
    pct   = count / max(1, len(slots_for_stats)) * 100
    bar   = "█" * int(pct / 2)
    print(f"  {key}: {count:>5}  ({pct:4.1f}%)  {bar}")

if answer_dist:
    max_share = max(answer_dist.values()) / max(1, len(slots_for_stats))
    flag = "✓ balanced" if max_share < 0.35 else "⚠ WARNING — positional bias detected"
    print(f"  Max share: {max_share:.1%}  {flag}")

print()
print("── Enrichment Coverage ─────────────────────────────────")
print(f"  Records with keywords      : {has_kw:>5} / {len(slots_for_stats)} ({has_kw/max(1,len(slots_for_stats)):.1%})")
print(f"  Records with context fringe: {has_fringe:>5} / {len(slots_for_stats)} ({has_fringe/max(1,len(slots_for_stats)):.1%})")

Total SFT records   : 2,921

── Bloom Level Distribution ────────────────────────────
  remember    :   116  ( 4.0%)  █
  understand  :   650  (22.3%)  ███████████
  apply       :   803  (27.5%)  █████████████
  analyze     :   737  (25.2%)  ████████████
  evaluate    :   409  (14.0%)  ███████
  create      :   206  ( 7.1%)  ███

── Chunk Type Distribution ─────────────────────────────
  definition  :   930  (31.8%)
  process     :   772  (26.4%)
  comparison  :   905  (31.0%)
  example     :   278  ( 9.5%)
  argument    :    36  ( 1.2%)

── Answer Key Distribution (should be ~25% each) ───────
  A:   742  (25.4%)  ████████████
  B:   729  (25.0%)  ████████████
  C:   739  (25.3%)  ████████████
  D:   711  (24.3%)  ████████████
  Max share: 25.4%  ✓ balanced

── Enrichment Coverage ─────────────────────────────────
  Records with keywords      :  2921 / 2921 (100.0%)
  Records with context fringe:  2921 / 2921 (100.0%)


In [18]:
# ── Inspect a random Option A sample ─────────────────────────────────────────
sample_meta = random.choice(sft_records)

print("=" * 70)
print(f"Sample ID: {sample_meta['id']}")
print("=" * 70)

for turn in sample_meta["conversations"]:
    role    = turn["role"].upper()
    content = turn["content"]
    # Truncate long turns for readability
    if len(content) > 600 and role == "USER":
        content = content[:]
    print(f"\n[{role}]\n{content}")

print("\n" + "=" * 70)

Sample ID: foundation_models_for_nlp_330_slot0_meta

[SYSTEM]
You are an expert educational content designer for Bloom's Taxonomy-aligned assessment.
Generate multiple-choice questions from educational text.
Return ONLY valid JSON. No markdown fences, preambles, or commentary.

## BLOOM LEVELS
remember: recall facts | understand: explain meaning | apply: use in scenario
analyze: compare/reason | evaluate: justify decision | create: design/propose

## CHUNK TYPE FORMS
definition: explain concept | process: steps/sequence | comparison: differences
example: real case | argument: evaluate claim

## GROUNDING (highest priority)
- Correct answer MUST be explicitly stated in or directly inferable from the chunk.
- Do NOT use external knowledge.

## MCQ RULES
- Exactly 4 options: A, B, C, D. Exactly ONE correct answer.
- Forbidden: 'All of the above', 'None of the above', 'Both A and B'.
- Options must be similar in length (±25-30%) and share the same grammatical structure.
- Options must be m

In [19]:
# ── Schema validation: verify every record has the expected structure ──────────
errors: List[str] = []

for path, label in [(SFT_META_PATH, "Option A")]:
    records = load_jsonl(path)
    for i, rec in enumerate(records):
        rid = rec.get("id", f"record_{i}")

        # Top-level fields
        if "id" not in rec:
            errors.append(f"{label} record {i}: missing 'id'")
        if "conversations" not in rec or not isinstance(rec["conversations"], list):
            errors.append(f"{label} {rid}: missing or invalid 'conversations'")
            continue

        convs = rec["conversations"]
        if len(convs) != 3:
            errors.append(f"{label} {rid}: expected 3 turns, got {len(convs)}")
            continue

        roles = [c.get("role") for c in convs]
        if roles != ["system", "user", "assistant"]:
            errors.append(f"{label} {rid}: wrong roles {roles}")

        # Validate assistant turn is parseable JSON with required MCQ fields
        assistant_content = convs[2].get("content", "")
        try:
            mcq_parsed = json.loads(assistant_content)
            for field in ["question", "options", "answer", "explanation"]:
                if field not in mcq_parsed:
                    errors.append(f"{label} {rid}: assistant JSON missing '{field}'")
            if set(mcq_parsed.get("options", {}).keys()) != {"A", "B", "C", "D"}:
                errors.append(f"{label} {rid}: options keys are not exactly A,B,C,D")
            if mcq_parsed.get("answer") not in {"A", "B", "C", "D"}:
                errors.append(f"{label} {rid}: invalid answer key")
        except json.JSONDecodeError as exc:
            errors.append(f"{label} {rid}: assistant content is not valid JSON: {exc}")

        # Confirm excluded fields are absent from the top-level record
        for forbidden in ["file_id", "s5_validation", "chunk_type_confidence",
                          "stage_name", "stage_number", "timestamp", "validation_result"]:
            if forbidden in rec:
                errors.append(f"{label} {rid}: forbidden field '{forbidden}' present")

if errors:
    print(f"⚠  {len(errors)} validation error(s) found:")
    for e in errors[:20]:
        print(f"   {e}")
    if len(errors) > 20:
        print(f"   ... and {len(errors) - 20} more.")
else:
    print(f"✓  All records passed schema validation.")
    print(f"   Option A: {len(load_jsonl(SFT_META_PATH)):,} records — {SFT_META_PATH}")

✓  All records passed schema validation.
   Option A: 2,921 records — /kaggle/working/GP_Quiz-Generation_Data/sft_metadata_conditioned.jsonl


## 10. Final SFT Sample Structure Reference

The cell below prints the exact schema of a complete SFT record
for documentation and thesis reference.

In [20]:
SCHEMA_REFERENCE = """
╔══════════════════════════════════════════════════════════════════════════╗
║           FINAL SFT RECORD SCHEMA  (LLaMA-Factory sharegpt format)       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Top-level fields:                                                       ║
║    id            : "<chunk_id>_slot<N>_meta"   (Option A)                ║
║    conversations : list of 3 turn dicts                                  ║
║                                                                          ║
║  Turn 1 — system                                                         ║
║    role    : "system"                                                    ║
║    content : static SFT_SYSTEM_PROMPT (same for all records)             ║
║                                                                          ║
║  Turn 2 — user  (Option A: conditioned)                                  ║
║    role    : "user"                                                      ║
║    content : ## CHUNK TEXT                                               ║
║              <source chunk text>                                         ║
║              ## CONTEXT  (if fringe available)                           ║
║              Previous: <prev_sentence>                                   ║
║              Next    : <next_sentence>                                   ║
║              ## METADATA                                                 ║
║              - chunk_type : <type>                                       ║
║              - keywords   : <kw1, kw2, ...>                              ║
║              [BLOOM LEVEL]: <level>                                      ║
║              [CHUNK TYPE]: <type>                                        ║
║              [FOCUS CONCEPT]: <concept>                                  ║
║              CHUNK TYPE INSTRUCTION: <style instruction>                 ║
║              ## YOUR TASK FOR THIS QUESTION                              ║
║              <task instruction>                                          ║
║                                                                          ║
║    role    : "user"                                                      ║
║    content : ## CHUNK TEXT + ## CONTEXT + ## METADATA + ## YOUR TASK     ║
║              (identical structure, NO [BLOOM LEVEL]/[FOCUS CONCEPT] tags)║
║                                                                          ║
║  Turn 3 — assistant  (identical for both options)                        ║
║    role    : "assistant"                                                 ║
║    content : clean JSON object (no markdown fences, no array wrapper):   ║
║              {                                                           ║
║                "question":    "<question stem>",                         ║
║                "options":     {"A": "...", "B": "...",                   ║
║                                "C": "...", "D": "..."},                  ║
║                "answer":      "<A|B|C|D>",                               ║
║                "explanation": "<1-2 sentence justification>"             ║
║              }                                                           ║
║                                                                          ║
║  EXCLUDED from all records:                                              ║
║    file_id, s5_validation, chunk_type_confidence,                        ║
║    stage_name, stage_number, timestamp, validation_result, scores        ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
"""

print(SCHEMA_REFERENCE)


╔══════════════════════════════════════════════════════════════════════════╗
║           FINAL SFT RECORD SCHEMA  (LLaMA-Factory sharegpt format)       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  Top-level fields:                                                       ║
║    id            : "<chunk_id>_slot<N>_meta"   (Option A)                ║
║    conversations : list of 3 turn dicts                                  ║
║                                                                          ║
║  Turn 1 — system                                                         ║
║    role    : "system"                                                    ║
║    content : static SFT_SYSTEM_PROMPT (same for all records)             ║
║                                                                          ║
║  Turn 2 — user  (Option A: conditioned)                                  

## 11. Fine-Tuning with LLaMA-Factory (LoRA / SFT)

This section fine-tunes **Qwen2.5-1.5B-Instruct** on the `sft_metadata_conditioned.jsonl`
dataset produced in Section 7, using [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
with LoRA adapters.

### What is reused from the reference notebook (verbatim workflow)
| Step | Reference → This notebook |
|---|---|
| LLaMA-Factory install | identical `pip install -e .` |
| `dataset_info.json` registration | adapted for `sharegpt` format |
| LoRA YAML config | adapted keys kept, dataset names updated |
| Training execution via `llamafactory-cli` | identical invocation |
| Adapter loading with `load_adapter()` | identical |
| `generate_resp()` inference helper | identical structure |

### What was adapted (because the tasks differ)
| Concern | Reference | This notebook |
|---|---|---|
| Dataset format | `alpaca` (instruction/input/output) | `sharegpt` (conversations list) — **unchanged** |
| `dataset_info.json` columns | `prompt`, `query`, `response` | `messages` → `conversations` with role/content tags |
| System prompt field | top-level `"system"` key | embedded as first turn in `conversations` |
| Assistant output | fenced JSON + explanation | clean JSON MCQ object |

### Dataset compatibility verdict
Your `sft_metadata_conditioned.jsonl` is **fully LLaMA-Factory–compatible as-is**.
The `sharegpt` format with `conversations` is natively supported. The `dataset_info.json`
generated in Section 8 already has the correct column mapping. No schema changes needed.

### Recommended pre-training checks (run Section 11.1 below)
1. Confirm `cutoff_len` covers the longest tokenised sample (measured automatically).
2. Confirm the JSONL file is on Google Drive and the path in the YAML is correct.
3. Confirm W&B API key is set in Colab secrets (optional but recommended).


## 11. Fine-Tuning with LLaMA-Factory (LoRA / SFT)

This section fine-tunes **Qwen2.5-1.5B-Instruct** on the `sft_metadata_conditioned.jsonl`
dataset produced in Section 7, using [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory)
with LoRA adapters.

### What is reused from the reference notebook (verbatim workflow)
| Step | Reference → This notebook |
|---|---|
| LLaMA-Factory install | identical `pip install -e .` |
| `dataset_info.json` registration | adapted for `sharegpt` format |
| LoRA YAML config | adapted keys kept, dataset names updated |
| Training execution via `llamafactory-cli` | identical invocation |
| Adapter loading with `load_adapter()` | identical |
| `generate_resp()` inference helper | identical structure |

### What was adapted (because the tasks differ)
| Concern | Reference | This notebook |
|---|---|---|
| Dataset format | `alpaca` (instruction/input/output) | `sharegpt` (conversations list) — **unchanged** |
| `dataset_info.json` columns | `prompt`, `query`, `response` | `messages` → `conversations` with role/content tags |
| System prompt field | top-level `"system"` key | embedded as first turn in `conversations` |
| Assistant output | fenced JSON + explanation | clean JSON MCQ object |

### Dataset compatibility verdict
Your `sft_metadata_conditioned.jsonl` is **fully LLaMA-Factory–compatible as-is**.
The `sharegpt` format with `conversations` is natively supported. The `dataset_info.json`
generated in Section 8 already has the correct column mapping. No schema changes needed.

### Recommended pre-training checks (run Section 11.1 below)
1. Confirm `cutoff_len` covers the longest tokenised sample (measured automatically).
2. Confirm the JSONL file is on Google Drive and the path in the YAML is correct.
3. Confirm W&B API key is set in Colab secrets (optional but recommended).


### 11.A  Authentication — Kaggle Secrets

Set these in Kaggle → **Settings → Add-ons → Secrets** before running:

| Secret Name | Value |
|---|---|
| `HF_TOKEN` | Your Hugging Face write token |
| `WANDB_API_KEY` | Your Weights & Biases API key |
| `HF_USERNAME` | Your HF username (e.g. `Aya-khaled`) |

These replace `google.colab.userdata` — Kaggle provides `kaggle_secrets.UserSecretsClient`.


In [21]:
# ─── Authentication: HF + W&B (Kaggle Secrets) ────────────────────────────────
import os
from huggingface_hub import login, create_repo

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    HF_TOKEN       = _s.get_secret("HF_TOKEN")
    WANDB_API_KEY  = _s.get_secret("WANDB_API_KEY")
    HF_USERNAME    = _s.get_secret("HF_USERNAME")   # e.g. "Aya-khaled"
    print("✓ All secrets loaded from Kaggle Secrets.")
except Exception as _e:
    HF_TOKEN      = os.environ.get("HF_TOKEN", "")
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
    HF_USERNAME   = os.environ.get("HF_USERNAME", "Aya-khaled")
    print(f"  Secrets fallback: {_e}")

os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "llamafactory-mcq-bloom"

# ── Login to HF Hub ───────────────────────────────────────────────────────────
login(token=HF_TOKEN)
print(f"✓ Logged in to Hugging Face as: {HF_USERNAME}")

# ── Create HF repos ───────────────────────────────────────────────────────────
LORA_REPO_ID   = f"{HF_USERNAME}/mcq-bloom-qwen-lora_v2"
MERGED_REPO_ID = f"{HF_USERNAME}/mcq-bloom-qwen-merged_v2"
CKPT_REPO_ID   = f"{HF_USERNAME}/mcq-bloom-qwen-checkpoints_v2"

for repo_id in [LORA_REPO_ID, MERGED_REPO_ID, CKPT_REPO_ID]:
    create_repo(repo_id, private=True, exist_ok=True)
    print(f"  Repo ready: {repo_id}")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ All secrets loaded from Kaggle Secrets.
✓ Logged in to Hugging Face as: Aya-khaled
  Repo ready: Aya-khaled/mcq-bloom-qwen-lora_v2
  Repo ready: Aya-khaled/mcq-bloom-qwen-merged_v2
  Repo ready: Aya-khaled/mcq-bloom-qwen-checkpoints_v2


### 11.1  Pre-flight Checks

Before writing any files we:
1. Verify the SFT JSONL exists and is non-empty
2. Compute token-length statistics → sets `CUTOFF_LEN` automatically
3. Audit Bloom-level distribution (needed for stratified split)
4. Audit answer-key distribution (detect positional bias A/B/C/D)


In [22]:
import json, os, random
from collections import Counter, defaultdict
from os.path import join

# ── Paths (Kaggle) ────────────────────────────────────────────────────────────
LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"

# Re-define for self-contained re-runs (in case cell 2 wasn't run in this session)
if "INPUT_DIR" not in dir():
    INPUT_DIR  = "/kaggle/input/datasets/ayakhaled51/mcqs-files"
if "OUTPUT_DIR" not in dir():
    OUTPUT_DIR = "/kaggle/working/GP_Quiz-Generation_Data"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# SFT file is a *generated output* — it lives in OUTPUT_DIR (writable)
SFT_META_PATH = join(OUTPUT_DIR, "sft_metadata_conditioned.jsonl")
FINETUNE_DIR  = join(OUTPUT_DIR, "llamafactory-finetune-data")
os.makedirs(FINETUNE_DIR, exist_ok=True)

assert os.path.exists(SFT_META_PATH), (
    f"SFT file not found: {SFT_META_PATH}\n"
    f"Run sections 3–7 first to generate the SFT JSONL from the input dataset."
)

with open(SFT_META_PATH, encoding="utf-8") as f:
    ALL_RECORDS = [json.loads(l) for l in f if l.strip()]

print(f"Total SFT records loaded: {len(ALL_RECORDS):,}")


Total SFT records loaded: 2,921


In [23]:
# ── Token-length audit (4 chars ≈ 1 token heuristic) ─────────────────────────
def char_len(rec):
    return sum(len(t.get("content", "")) for t in rec.get("conversations", []))

tok_lens  = [char_len(r) // 4 for r in ALL_RECORDS]
sorted_tl = sorted(tok_lens)
n = len(sorted_tl)

p50 = sorted_tl[n // 2]
p90 = sorted_tl[int(0.90 * n)]
p95 = sorted_tl[int(0.95 * n)]
p99 = sorted_tl[int(0.99 * n)]

CUTOFF_LEN = min(4096, ((p95 // 512) + 1) * 512)
truncated  = sum(1 for t in tok_lens if t > CUTOFF_LEN)

print("Token length statistics (4-char heuristic):")
print(f"  Min  : {min(tok_lens):>6,}")
print(f"  P50  : {p50:>6,}")
print(f"  P90  : {p90:>6,}")
print(f"  P95  : {p95:>6,}")
print(f"  P99  : {p99:>6,}")
print(f"  Max  : {max(tok_lens):>6,}")
print(f"\nCUTOFF_LEN = {CUTOFF_LEN}  (P95 rounded to next 512, max 4096)")
print(f"Records that will be truncated: {truncated} ({truncated/n:.1%})")


Token length statistics (4-char heuristic):
  Min  :    878
  P50  :  1,108
  P90  :  1,249
  P95  :  1,286
  P99  :  1,360
  Max  :  1,491

CUTOFF_LEN = 1536  (P95 rounded to next 512, max 4096)
Records that will be truncated: 0 (0.0%)


In [24]:
def get_bloom(rec):
    for turn in rec.get("conversations", []):
        if turn.get("role") == "user":
            for line in turn["content"].splitlines():
                if line.strip().startswith("[BLOOM LEVEL]"):
                    return line.split(":", 1)[-1].strip().lower()
    return "unknown"

bloom_labels = [get_bloom(r) for r in ALL_RECORDS]
bloom_dist   = Counter(bloom_labels)

print("Bloom level distribution:")
for lvl in ["remember","understand","apply","analyze","evaluate","create"]:
    c   = bloom_dist.get(lvl, 0)
    bar = "█" * int(c / max(bloom_dist.values(), default=1) * 30)
    print(f"  {lvl:<12}: {c:>5}  ({c/len(ALL_RECORDS):.1%})  {bar}")


Bloom level distribution:
  remember    :   116  (4.0%)  ████
  understand  :   650  (22.3%)  ████████████████████████
  apply       :   803  (27.5%)  ██████████████████████████████
  analyze     :   737  (25.2%)  ███████████████████████████
  evaluate    :   409  (14.0%)  ███████████████
  create      :   206  (7.1%)  ███████


In [25]:
def get_answer(rec):
    for turn in rec.get("conversations", []):
        if turn.get("role") == "assistant":
            try:
                return json.loads(turn["content"]).get("answer", "?")
            except Exception:
                return "?"
    return "?"

answer_labels = [get_answer(r) for r in ALL_RECORDS]
ans_dist = Counter(answer_labels)

print("Answer distribution:")
for k in ["A","B","C","D"]:
    c = ans_dist.get(k, 0)
    print(f"  {k}: {c:>5}  ({c/len(ALL_RECORDS):.1%})")

max_share = max(ans_dist.get(k,0) for k in ["A","B","C","D"]) / len(ALL_RECORDS)
if max_share > 0.40:
    print("\n  WARNING: Positional bias detected (one answer key > 40%). Run debiasing cell below.")
else:
    print(f"\n  OK: Answer distribution balanced (max share {max_share:.1%}). No debiasing needed.")


Answer distribution:
  A:   742  (25.4%)
  B:   729  (25.0%)
  C:   739  (25.3%)
  D:   711  (24.3%)

  OK: Answer distribution balanced (max share 25.4%). No debiasing needed.


In [26]:
# ── (Optional) Answer-key debiasing ──────────────────────────────────────────
DEBIAS = False

if DEBIAS:
    by_answer = defaultdict(list)
    for rec, ans in zip(ALL_RECORDS, answer_labels):
        by_answer[ans].append(rec)
    cap = min(len(by_answer[k]) for k in ["A","B","C","D"]) * 2
    debiased = []
    rng = random.Random(42)
    for k in ["A","B","C","D"]:
        bucket = by_answer[k][:]
        rng.shuffle(bucket)
        debiased.extend(bucket[:cap])
    debiased.extend(by_answer.get("?", []))
    print(f"Before debiasing: {len(ALL_RECORDS):,}")
    ALL_RECORDS  = debiased
    bloom_labels = [get_bloom(r) for r in ALL_RECORDS]
    print(f"After debiasing : {len(ALL_RECORDS):,}")
else:
    print("Debiasing skipped.")


Debiasing skipped.


### 11.2  Create Train / Val / Test Splits

**Split ratios:** 85% train · 10% val · 5% test — **stratified by Bloom level**.


In [29]:
TRAIN_RATIO = 0.90
VAL_RATIO   = 0.10
SEED        = 42

by_bloom = defaultdict(list)
for rec, lvl in zip(ALL_RECORDS, bloom_labels):
    by_bloom[lvl].append(rec)

train_recs, val_recs = [], []
rng = random.Random(SEED)

for lvl, recs in by_bloom.items():
    rng.shuffle(recs)
    n = len(recs)
    n_val   = max(1, int(n * VAL_RATIO))
    n_train = n - n_val
    train_recs.extend(recs[:n_train])
    val_recs.extend(recs[n_train : n_train + n_val])
    
rng.shuffle(train_recs); rng.shuffle(val_recs)

total = len(train_recs) + len(val_recs)
assert total == len(ALL_RECORDS), "Split sum mismatch!"

print(f"Train : {len(train_recs):,}  ({len(train_recs)/len(ALL_RECORDS):.1%})")
print(f"Val   : {len(val_recs):,}  ({len(val_recs)/len(ALL_RECORDS):.1%})")
print(f"Total : {total:,}  OK")


Train : 2,632  (90.1%)
Val   : 289  (9.9%)
Total : 2,921  OK


In [30]:
TRAIN_PATH = join(FINETUNE_DIR, "mcq_sft_train.jsonl")
VAL_PATH   = join(FINETUNE_DIR, "mcq_sft_val.jsonl")

def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  Written: {path}  ({len(records):,} records)")

write_jsonl(TRAIN_PATH, train_recs)
write_jsonl(VAL_PATH,   val_recs)
print("\nAll split files written.")


  Written: /kaggle/working/GP_Quiz-Generation_Data/llamafactory-finetune-data/mcq_sft_train.jsonl  (2,632 records)
  Written: /kaggle/working/GP_Quiz-Generation_Data/llamafactory-finetune-data/mcq_sft_val.jsonl  (289 records)

All split files written.


In [32]:
print(f"{'Level':<12}  {'Train%':>7}  {'Val%':>7}")
for lvl in sorted(set(bloom_labels)):
    tr = sum(1 for r in train_recs if get_bloom(r) == lvl) / max(1, len(train_recs))
    va = sum(1 for r in val_recs   if get_bloom(r) == lvl) / max(1, len(val_recs))
    print(f"  {lvl:<12}  {tr:>6.1%}  {va:>6.1%}")
print("\n(Values should be within ~2% across splits.)")


Level          Train%     Val%
  analyze        25.2%   25.3%
  apply          27.5%   27.7%
  create          7.1%    6.9%
  evaluate       14.0%   13.8%
  remember        4.0%    3.8%
  understand     22.2%   22.5%

(Values should be within ~2% across splits.)


### 11.3  Register Datasets in LLaMA-Factory

In [33]:
import json, os, subprocess, sys

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"

# ── Step 1: Clone LLaMA-Factory if not already present ───────────────────────
if not os.path.exists(LLAMA_FACTORY_DIR):
    print("LLaMA-Factory not found — cloning...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/hiyouga/LLaMA-Factory.git",
         LLAMA_FACTORY_DIR],
        check=True
    )
    print("✓ Cloned.")
else:
    print("✓ LLaMA-Factory already present — skipping clone.")

# ── Step 2: Install dependencies if llamafactory-cli is missing ───────────────
result = subprocess.run(["which", "llamafactory-cli"], capture_output=True)
if result.returncode != 0:
    print("Installing LLaMA-Factory dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[torch,metrics]", "-q"],
        cwd=LLAMA_FACTORY_DIR, check=True
    )
    print("✓ Dependencies installed.")
else:
    print("✓ llamafactory-cli already available.")

LLaMA-Factory not found — cloning...


Cloning into '/kaggle/working/LLaMA-Factory'...


✓ Cloned.
Installing LLaMA-Factory dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.1 MB/s eta 0:00:00
✓ Dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [34]:
import json, os

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"
DATASET_INFO_PATH = os.path.join(LLAMA_FACTORY_DIR, "data/dataset_info.json")

with open(DATASET_INFO_PATH, encoding="utf-8") as f:
    dataset_info = json.load(f)

sharegpt_tags = {
    "role_tag"     : "role",
    "content_tag"  : "content",
    "user_tag"     : "user",
    "assistant_tag": "assistant",
    "system_tag"   : "system",
}

mcq_entries = {
    "mcq_sft_train": {"file_name": TRAIN_PATH, "formatting": "sharegpt",
                      "columns": {"messages": "conversations"}, "tags": sharegpt_tags},
    "mcq_sft_val"  : {"file_name": VAL_PATH,   "formatting": "sharegpt",
                      "columns": {"messages": "conversations"}, "tags": sharegpt_tags},}

dataset_info.update(mcq_entries)

with open(DATASET_INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print("dataset_info.json updated. Registered:")
for k, v in mcq_entries.items():
    print(f"  {k}: {v['file_name']}")


dataset_info.json updated. Registered:
  mcq_sft_train: /kaggle/working/GP_Quiz-Generation_Data/llamafactory-finetune-data/mcq_sft_train.jsonl
  mcq_sft_val: /kaggle/working/GP_Quiz-Generation_Data/llamafactory-finetune-data/mcq_sft_val.jsonl


In [38]:
# ── LLaMA-Factory schema smoke-test (SAFE VERSION) ───────────────────────────
import subprocess, sys, os

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"

script = f'''
import sys, os
os.chdir("{LLAMA_FACTORY_DIR}")
sys.path.insert(0, "{LLAMA_FACTORY_DIR}/src")

try:
    from transformers import Seq2SeqTrainingArguments

    from llamafactory.hparams import (
        ModelArguments,
        DataArguments,
        FinetuningArguments,
    )

    from llamafactory.data import (
        get_dataset,
        get_template_and_fix_tokenizer
    )

    from llamafactory.model import load_tokenizer

    # ── Build args manually (NO distributed checks) ─────────────────

    model_args = ModelArguments(
        model_name_or_path="/kaggle/working/qwen_model"
    )

    data_args = DataArguments(
        dataset="mcq_sft_train",
        template="qwen",
        cutoff_len=512,
        max_samples=2
    )

    training_args = Seq2SeqTrainingArguments(
        output_dir="/tmp/smoke",
        per_device_train_batch_size=1,
        do_train=False,
        do_eval=False,
        report_to=[]
    )

    finetuning_args = FinetuningArguments(
        finetuning_type="lora"
    )

    # ── Load tokenizer ───────────────────────────────────────────────

    tokenizer_module = load_tokenizer(model_args)
    tokenizer = tokenizer_module["tokenizer"]

    template = get_template_and_fix_tokenizer(
        tokenizer,
        data_args
    )

    # ── Load dataset ─────────────────────────────────────────────────

    dataset_module = get_dataset(
        template=template,
        model_args=model_args,
        data_args=data_args,
        training_args=training_args,
        tokenizer=tokenizer,
        processor=None,
        stage="sft"
    )

    train_dataset = dataset_module["train_dataset"]

    print("Train samples:", len(train_dataset))
    print("SCHEMA OK")

except Exception as e:
    import traceback
    traceback.print_exc()
    print("SCHEMA ERROR:", repr(e))
'''

result = subprocess.run(
    [sys.executable, "-c", script],
    capture_output=True,
    text=True,
    cwd=LLAMA_FACTORY_DIR
)

print(result.stdout[-4000:])

if result.stderr:
    print("STDERR:")
    print(result.stderr[-4000:])

 current system's error (e.g., 2%) provides better tools for continued improvement."
}<|im_end|>

label_ids:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -1

### 11.4  Write LoRA Training YAML (Kaggle-Optimized)

**Key improvements over original:**

| Parameter | Before | After | Reason |
|---|---|---|---|
| `output_dir` | Drive path | `/kaggle/working/lora-adapter` | Kaggle-local; pushed to HF after each checkpoint |
| `push_to_hub` | true (checkpoints to Drive) | true → HF Hub | HF = checkpoint backup; no Drive bloat |
| `hub_strategy` | `checkpoint` | `all_checkpoints` | Every checkpoint auto-pushed to HF |
| `save_total_limit` | 1 | **3** | Keep 3 local + all on HF |
| `load_best_model_at_end` | false | **true** | Auto-load best val-loss checkpoint |
| `metric_for_best_model` | *(missing)* | `eval_loss` | Correct metric for generative tasks |
| `greater_is_better` | *(missing)* | `false` | Lower loss = better |
| `early_stopping_patience` | *(missing)* | **5** | Stop if no improvement for 5 evals |
| `logging_steps` | 4 | **10** | Cleaner console output |
| `eval_steps` | 24 | **50** | Evaluate every ~50 steps |
| `save_steps` | 46 | **50** | Align with eval for best-checkpoint tracking |
| `neftune_noise_alpha` | 5 | **5** | ✓ kept — helps instruction following |


In [81]:
import os
from os.path import join

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"

# ── Local Kaggle output dir (temporary — pushed to HF, NOT kept on Kaggle) ───
KAGGLE_LORA_DIR = "/kaggle/working/mcq-lora-adapter_v2"
os.makedirs(KAGGLE_LORA_DIR, exist_ok=True)

# Final artifacts stay in /kaggle/working and are pushed to HF Hub.
# No Google Drive — Drive is not supported on Kaggle.
KAGGLE_BEST_LORA   = "/kaggle/working/mcq-lora-adapter-best_v2"
KAGGLE_BEST_MERGED = "/kaggle/working/mcq-qwen-merged-best_v2"
os.makedirs(KAGGLE_BEST_LORA,   exist_ok=True)
os.makedirs(KAGGLE_BEST_MERGED, exist_ok=True)

YAML_PATH = os.path.join(LLAMA_FACTORY_DIR, "examples/train_lora/mcq_sft_finetune.yaml")
os.makedirs(os.path.dirname(YAML_PATH), exist_ok=True)

cutoff = CUTOFF_LEN   # from 11.1

yaml_content = f"""
### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true
flash_attn: sdpa                  # Use fa2 if flash-attn is compiled; sdpa otherwise

### method
stage: sft
do_train: true
finetuning_type: lora

### LoRA settings
lora_rank: 16                     # ↓ from 64. ~4x fewer params → less overfitting risk
lora_alpha: 16                    # scale=1.0. Conservative for small dataset.
lora_dropout: 0.05                # Fine at r=16
lora_target: all                  # Keep all layers for factual+format learning

### dataset
dataset: mcq_sft_train
eval_dataset: mcq_sft_val
template: qwen
cutoff_len: {cutoff}                 # ↑ to cover full sequences (800-1200 tokens + prompt)
train_on_prompt: false
packing: false
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: {KAGGLE_LORA_DIR}
logging_strategy: steps
save_strategy: steps
eval_strategy: steps
logging_steps: 10
save_steps: 164                 # ← sync with eval_steps for best-checkpoint capture
eval_steps: 82
save_total_limit: 5               # ↑ slightly so best checkpoint isn't evicted
plot_loss: true
include_num_input_tokens_seen: true

### best-model tracking
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5             # ↓ from 1e-4. Key anti-forgetting change.
num_train_epochs: 2.0             # ↓ from 3. Rely on load_best_model_at_end.
lr_scheduler_type: cosine
warmup_ratio: 0.05                # ↓ from 0.1. Sufficient for gradient shock prevention.
weight_decay: 0.01                # ADD: L2 regularization
bf16: true
ddp_timeout: 180000000
neftune_noise_alpha: 10           # ↑ from 5. More embedding regularization.

### eval
per_device_eval_batch_size: 1

### Hugging Face Hub
push_to_hub: false
export_hub_model_id: {LORA_REPO_ID}
hub_private_repo: true
hub_strategy: all_checkpoints

### tracking
report_to: ["wandb"]
run_name: mcq-bloom-qwen-v2-optimized

### resume (uncomment to resume from a specific checkpoint)
# resume_from_checkpoint: {KAGGLE_LORA_DIR}/checkpoint-658
"""

with open(YAML_PATH, "w") as f:
    f.write(yaml_content)

print(f"Training YAML written → {YAML_PATH}")
print()
print(yaml_content)


Training YAML written → /kaggle/working/LLaMA-Factory/examples/train_lora/mcq_sft_finetune.yaml


### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true
flash_attn: sdpa                  # Use fa2 if flash-attn is compiled; sdpa otherwise

### method
stage: sft
do_train: true
finetuning_type: lora

### LoRA settings
lora_rank: 16                     # ↓ from 64. ~4x fewer params → less overfitting risk
lora_alpha: 16                    # scale=1.0. Conservative for small dataset.
lora_dropout: 0.05                # Fine at r=16
lora_target: all                  # Keep all layers for factual+format learning

### dataset
dataset: mcq_sft_train
eval_dataset: mcq_sft_val
template: qwen
cutoff_len: 1536                 # ↑ to cover full sequences (800-1200 tokens + prompt)
train_on_prompt: false
packing: false
overwrite_cache: true
preprocessing_num_workers: 8

### output
output_dir: /kaggle/working/mcq-lora-adapter_v2
logging_strategy: steps
save_strategy: steps
e

### 11.5  Pretty Training Progress Callback

A custom `TrainerCallback` that prints a structured, human-readable progress line
after every logging step and evaluation — visible in Kaggle's notebook output.

```
Step  120/500 | Epoch 0.75 | TrainLoss 1.245 | EvalLoss 1.102 | LR 2.1e-5 | GradNorm 0.84 | ETA 00:42:11
```


In [43]:
import time
from transformers import TrainerCallback

class PrettyProgressCallback(TrainerCallback):
    """
    Prints a structured progress line to console after every log/eval event.
    Compatible with LLaMA-Factory's Trainer (which uses HuggingFace Trainer).
    """
    def __init__(self):
        self._train_start = None
        self._best_eval_loss = float("inf")
        self._best_step = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start = time.time()
        total = state.max_steps
        print("=" * 80)
        print(f"  TRAINING START  |  Total steps: {total}  |  Epochs: {args.num_train_epochs}")
        print("=" * 80)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step      = state.global_step
        total     = state.max_steps
        epoch     = state.epoch or 0.0
        elapsed   = time.time() - (self._train_start or time.time())
        eta_secs  = (elapsed / max(step, 1)) * max(total - step, 0)
        eta_str   = time.strftime("%H:%M:%S", time.gmtime(eta_secs))

        train_loss = logs.get("loss", logs.get("train_loss", None))
        eval_loss  = logs.get("eval_loss", None)
        lr         = logs.get("learning_rate", None)
        grad_norm  = logs.get("grad_norm", None)

        parts = [
            f"Step {step:>5}/{total}",
            f"Epoch {epoch:.2f}",
        ]
        if train_loss is not None:
            parts.append(f"TrainLoss {train_loss:.4f}")
        if eval_loss is not None:
            parts.append(f"EvalLoss {eval_loss:.4f}")
            if eval_loss < self._best_eval_loss:
                self._best_eval_loss = eval_loss
                self._best_step = step
                parts.append("⭐ NEW BEST")
        if lr is not None:
            parts.append(f"LR {lr:.2e}")
        if grad_norm is not None:
            parts.append(f"GradNorm {grad_norm:.3f}")
        parts.append(f"ETA {eta_str}")

        print(" | ".join(parts))

    def on_save(self, args, state, control, **kwargs):
        step = state.global_step
        print(f"  💾 Checkpoint saved at step {step} → {args.output_dir}/checkpoint-{step}")
        if self._best_step == step:
            print(f"  ⭐ This is the BEST checkpoint so far (eval_loss={self._best_eval_loss:.4f})")

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - (self._train_start or time.time())
        print("=" * 80)
        print(f"  TRAINING COMPLETE  |  Total time: {time.strftime('%H:%M:%S', time.gmtime(elapsed))}")
        if self._best_step:
            print(f"  Best checkpoint: step {self._best_step}  |  eval_loss={self._best_eval_loss:.4f}")
        print("=" * 80)

print("✓ PrettyProgressCallback defined.")


✓ PrettyProgressCallback defined.


### 11.6  Run LoRA Fine-Tuning

**What to watch for:**
- Loss should decrease for the first 20% of steps. Explosion → check `bf16`, `learning_rate`, dataset schema.
- Eval loss should track train loss (within ~0.3 nats). Eval rising while train falls = overfitting → early stopping will trigger.
- Checkpoints are automatically pushed to HF Hub — check your repo during training.
- Drive only receives the final best model (Section 11.9).

Expected duration on Kaggle T4: ~2–4 hours for 3 epochs over ~3000 records.


In [44]:
import os, subprocess, sys

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"
YAML_PATH = os.path.join(LLAMA_FACTORY_DIR, "examples/train_lora/mcq_sft_finetune.yaml")

# Set W&B env vars
os.environ["WANDB_PROJECT"] = "llamafactory-mcq-bloom"
# To disable W&B: os.environ["WANDB_MODE"] = "disabled"

print("Starting training...")
print(f"YAML: {YAML_PATH}")
print("-" * 60)

result = subprocess.run(
    ["llamafactory-cli", "train", YAML_PATH],
    cwd=LLAMA_FACTORY_DIR,
    env={**os.environ},
)

if result.returncode == 0:
    print("\n✓ Training complete.")
else:
    print(f"\n✗ Training exited with code {result.returncode}.")


Starting training...
YAML: /kaggle/working/LLaMA-Factory/examples/train_lora/mcq_sft_finetune.yaml
------------------------------------------------------------
[INFO|2026-05-23 16:35:55] llamafactory.launcher:144 >> Initializing 2 distributed tasks at: 127.0.0.1:46049


W0523 16:35:57.670000 381 torch/distributed/run.py:852] 
W0523 16:35:57.670000 381 torch/distributed/run.py:852] *****************************************
W0523 16:35:57.670000 381 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0523 16:35:57.670000 381 torch/distributed/run.py:852] *****************************************
[W523 16:36:13.038093726 ProcessGroupNCCL.cpp:929] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())
[W523 16:36:13.061409626 ProcessGroupNCCL.cpp:929] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|configuration_utils.py:665] 202

[INFO|2026-05-23 16:36:13] llamafactory.hparams.parser:144 >> Set `ddp_find_unused_parameters` to False in DDP training since LoRA is enabled.
[INFO|2026-05-23 16:36:13] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 2, device: cuda:0, distributed training: True, compute dtype: torch.bfloat16
[INFO|2026-05-23 16:36:13] llamafactory.hparams.parser:523 >> Process rank: 1, world size: 2, device: cuda:1, distributed training: True, compute dtype: torch.bfloat16


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|configuration_utils.py:665] 2026-05-23 16:36:15,110 >> loading configuration file /kaggle/working/qwen_model/config.json
[INFO|configuration_utils.py:739] 2026-05-23 16:36:15,111 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "fu

[INFO|2026-05-23 16:36:16] llamafactory.data.loader:144 >> Loading dataset /kaggle/working/GP_Quiz-Generation_Data/llamafactory-finetune-data/mcq_sft_train.jsonl...


...

{'train_runtime': '1.631e+04', 'train_samples_per_second': '0.323', 'train_steps_per_second': '0.04', 'train_loss': '0.6803', 'epoch': '2', 'num_input_tokens_seen': 5525520}


[INFO|trainer.py:4115] 2026-05-23 21:08:57,796 >> Saving model checkpoint to /kaggle/working/mcq-lora-adapter_v2
[INFO|configuration_utils.py:665] 2026-05-23 21:08:57,820 >> loading configuration file /kaggle/working/qwen_model/config.json
[INFO|configuration_utils.py:739] 2026-05-23 21:08:57,821 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",


wandb: 
wandb: 🚀 View run mcq-bloom-qwen-v2-optimized at: https://wandb.ai/ayaakhaled4356-no-company/llamafactory-mcq-bloom/runs/uj7rh2aj
wandb: Find logs at: wandb/run-20260523_163704-uj7rh2aj/logs


W0523 21:09:01.408000 381 torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 388 closing signal SIGTERM
E0523 21:09:01.773000 381 torch/distributed/elastic/multiprocessing/api.py:984] failed (exitcode: 1) local_rank: 0 (pid: 387) of binary: /usr/bin/python3
Traceback (most recent call last):
  File "/usr/local/bin/torchrun", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 362, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 991, in main
    run(args)
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/run.py", line 982, in run
    elastic_launch(
  File "/usr/local/lib/python3.12/dist-packages/torch/distributed/launcher/api.py", line 170, in __call__
    return launch_agent(self._config, self._entrypoint, list(args))
      


✗ Training exited with code 1.


In [84]:
# ── Resume training from a checkpoint ────────────────────────────────────────
# 1. Add this line to your YAML:
#    resume_from_checkpoint: /kaggle/working/mcq-lora-adapter/checkpoint-500
# 2. Uncomment and run this cell instead of the one above.

# import os, subprocess
# LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"
# YAML_PATH = os.path.join(LLAMA_FACTORY_DIR, "examples/train_lora/mcq_sft_finetune.yaml")
# subprocess.run(["llamafactory-cli", "train", YAML_PATH], cwd=LLAMA_FACTORY_DIR, env=os.environ)

### 11.7  Best Checkpoint Selection

`load_best_model_at_end: true` in the YAML automatically restores the best eval-loss checkpoint.
This cell additionally scans checkpoints for JSON parse rate (a task-specific metric that
complements eval loss) and identifies the overall best checkpoint for export.


In [85]:
import glob, os, json

KAGGLE_LORA_DIR = "/kaggle/working/mcq-lora-adapter_v2"

# ── Read trainer_state.json to find best checkpoint by eval_loss ──────────────
trainer_state_path = os.path.join(KAGGLE_LORA_DIR, "trainer_state.json")

BEST_CKPT = None
best_eval_loss = float("inf")

if os.path.exists(trainer_state_path):
    with open(trainer_state_path) as f:
        state = json.load(f)

    print("Checkpoint history (eval_loss):")
    print(f"  {'Step':>6}  {'Eval Loss':>10}  {'Best?':>6}")
    print("  " + "-" * 30)

    for entry in state.get("log_history", []):
        if "eval_loss" in entry:
            step      = entry["step"]
            eval_loss = entry["eval_loss"]
            is_best   = ""
            if eval_loss < best_eval_loss:
                best_eval_loss = eval_loss
                is_best = "⭐"
                BEST_CKPT = os.path.join(KAGGLE_LORA_DIR, f"checkpoint-{step}")
            print(f"  {step:>6}  {eval_loss:>10.4f}  {is_best:>6}")

    # LLaMA-Factory also stores best_model_checkpoint
    if state.get("best_model_checkpoint"):
        BEST_CKPT = state["best_model_checkpoint"]
        print(f"\n  best_model_checkpoint from trainer_state: {BEST_CKPT}")

if BEST_CKPT and os.path.exists(BEST_CKPT):
    print(f"\n✓ Best checkpoint: {BEST_CKPT}  (eval_loss={best_eval_loss:.4f})")
else:
    # Fallback: use the adapter dir itself (load_best_model_at_end already restored it)
    BEST_CKPT = KAGGLE_LORA_DIR
    print(f"\n  Using full adapter dir as best checkpoint: {BEST_CKPT}")



  Using full adapter dir as best checkpoint: /kaggle/working/mcq-lora-adapter_v2


### 11.8  Push All Checkpoints to Hugging Face Hub

Since `hub_strategy: all_checkpoints` is set in the YAML, LLaMA-Factory automatically pushes
every saved checkpoint to HF during training. This cell verifies and manually syncs any
checkpoints that may have been missed (e.g., after a session restart).

**Storage strategy:**
- ✅ Checkpoints → HF Hub (`CKPT_REPO_ID`) automatically during training
- ✅ Final best LoRA → HF Hub (`LORA_REPO_ID`) + `/kaggle/working/mcq-lora-adapter-best/`
- ✅ Final merged model → HF Hub (`MERGED_REPO_ID`) + `/kaggle/working/mcq-qwen-merged-best/`
- ❌ Checkpoints NOT kept permanently on Kaggle disk (deleted after HF push)


In [87]:
# import glob

# ckpts = glob.glob("/kaggle/working/mcq-lora-adapter_v2/checkpoint-*")

# for ckpt in ckpts:
#     readme_path = os.path.join(ckpt, "README.md")
    
#     if os.path.exists(readme_path):
#         with open(readme_path, "r") as f:
#             text = f.read()

#         text = text.replace(
#             "/kaggle/working/qwen_model",
#             "Qwen/Qwen2.5-1.5B-Instruct"
#         )

#         with open(readme_path, "w") as f:
#             f.write(text)

# print("Fixed README base_model paths")

Fixed README base_model paths


In [88]:
import glob, os
from huggingface_hub import HfApi

api = HfApi()
KAGGLE_LORA_DIR = "/kaggle/working/mcq-lora-adapter_v2"

# ── Sync any local checkpoints to HF Hub ─────────────────────────────────────
checkpoints = sorted(
    glob.glob(os.path.join(KAGGLE_LORA_DIR, "checkpoint-*")),
    key=lambda p: int(p.split("-")[-1])
)

print(f"Found {len(checkpoints)} local checkpoint(s):")
for ckpt in checkpoints:
    step = ckpt.split("-")[-1]
    ckpt_folder = f"checkpoint-{step}"
    print(f"  Uploading {ckpt_folder} → {CKPT_REPO_ID} ...", end=" ", flush=True)
    api.upload_folder(
        folder_path=ckpt,
        repo_id=CKPT_REPO_ID,
        repo_type="model",
        path_in_repo=ckpt_folder,
    )
    print("✓")

# ── Clean up local checkpoints to free Kaggle disk space ─────────────────────
import shutil
for ckpt in checkpoints:
    if ckpt != BEST_CKPT:   # keep best checkpoint locally for export
        shutil.rmtree(ckpt, ignore_errors=True)
        print(f"  Deleted local: {ckpt}")

print("\n✓ All checkpoints synced to HF Hub and local non-best copies removed.")
print(f"  View at: https://huggingface.co/{CKPT_REPO_ID}")


Found 5 local checkpoint(s):
  Uploading checkpoint-164 → Aya-khaled/mcq-bloom-qwen-checkpoints_v2 ... 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓
  Uploading checkpoint-328 → Aya-khaled/mcq-bloom-qwen-checkpoints_v2 ... 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

✓
  Uploading checkpoint-658 → Aya-khaled/mcq-bloom-qwen-checkpoints_v2 ... 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓
  Deleted local: /kaggle/working/mcq-lora-adapter_v2/checkpoint-164
  Deleted local: /kaggle/working/mcq-lora-adapter_v2/checkpoint-328
  Deleted local: /kaggle/working/mcq-lora-adapter_v2/checkpoint-492
  Deleted local: /kaggle/working/mcq-lora-adapter_v2/checkpoint-656
  Deleted local: /kaggle/working/mcq-lora-adapter_v2/checkpoint-658

✓ All checkpoints synced to HF Hub and local non-best copies removed.
  View at: https://huggingface.co/Aya-khaled/mcq-bloom-qwen-checkpoints_v2


### 11.9  Export & Merge Best LoRA → Merged Model

In [89]:
import os, shutil
from os.path import join

LLAMA_FACTORY_DIR  = "/kaggle/working/LLaMA-Factory"
KAGGLE_MERGED_DIR  = "/kaggle/working/mcq-qwen-merged_v2"
os.makedirs(KAGGLE_MERGED_DIR, exist_ok=True)

EXPORT_YAML_PATH = os.path.join(LLAMA_FACTORY_DIR, "examples/merge_lora/mcq_sft_export.yaml")
os.makedirs(os.path.dirname(EXPORT_YAML_PATH), exist_ok=True)

export_yaml = f"""
### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
finetuning_type: lora
adapter_name_or_path: {BEST_CKPT}

### export
export_dir: {KAGGLE_MERGED_DIR}
export_size: 2
export_device: cpu
export_legacy_format: false
"""

with open(EXPORT_YAML_PATH, "w") as f:
    f.write(export_yaml)

print(f"Export YAML → {EXPORT_YAML_PATH}")
print(export_yaml)


Export YAML → /kaggle/working/LLaMA-Factory/examples/merge_lora/mcq_sft_export.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
finetuning_type: lora
adapter_name_or_path: /kaggle/working/mcq-lora-adapter_v2

### export
export_dir: /kaggle/working/mcq-qwen-merged_v2
export_size: 2
export_device: cpu
export_legacy_format: false



In [98]:
import subprocess, os

LLAMA_FACTORY_DIR = "/kaggle/working/LLaMA-Factory"
EXPORT_YAML_PATH  = os.path.join(LLAMA_FACTORY_DIR, "examples/merge_lora/mcq_sft_export.yaml")

print("Merging LoRA adapter into base model...")
result = subprocess.run(
    ["llamafactory-cli", "export", EXPORT_YAML_PATH],
    cwd=LLAMA_FACTORY_DIR,
    env=os.environ,
)
if result.returncode == 0:
    print("✓ Export complete.")
else:
    print(f"✗ Export failed (code {result.returncode}).")


Merging LoRA adapter into base model...


KeyboardInterrupt: 

In [101]:
import subprocess

print("Merging LoRA adapter into base model...")

process = subprocess.Popen(
    ["llamafactory-cli", "export", EXPORT_YAML_PATH],
    cwd=LLAMA_FACTORY_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

Merging LoRA adapter into base model...
[INFO|configuration_utils.py:667] 2026-05-23 23:39:34,236 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:739] 2026-05-23 23:39:34,238 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
  

In [102]:
import os
KAGGLE_MERGED_DIR = "/kaggle/working/mcq-qwen-merged_v2"
files = os.listdir(KAGGLE_MERGED_DIR)
print(f"Files in merged model dir ({KAGGLE_MERGED_DIR}):")
for fname in sorted(files):
    size = os.path.getsize(os.path.join(KAGGLE_MERGED_DIR, fname)) // (1024**2)
    print(f"  {fname}  ({size} MB)")
assert any(f.endswith(".safetensors") for f in files), "No safetensors — export failed."
assert "config.json" in files, "config.json missing."
print("\n✓ Merged model export verified.")


Files in merged model dir (/kaggle/working/mcq-qwen-merged_v2):
  Modelfile  (0 MB)
  chat_template.jinja  (0 MB)
  config.json  (0 MB)
  generation_config.json  (0 MB)
  model-00001-of-00002.safetensors  (1899 MB)
  model-00002-of-00002.safetensors  (1044 MB)
  model.safetensors.index.json  (0 MB)
  tokenizer.json  (10 MB)
  tokenizer_config.json  (0 MB)

✓ Merged model export verified.


### 11.10  Copy Best Model + Best LoRA to a Clean Output Folder

Copies ONLY the final artifacts into clean `/kaggle/working` output folders:
- `/kaggle/working/mcq-lora-adapter-best/` — the best LoRA adapter files
- `/kaggle/working/mcq-qwen-merged-best/` — the final merged model (inference-ready)

These are then pushed to HF Hub in §11.11. No Google Drive is used.


In [103]:
import shutil, os

KAGGLE_LORA_DIR    = "/kaggle/working/mcq-lora-adapter_v2"
KAGGLE_MERGED_DIR  = "/kaggle/working/mcq-qwen-merged_v2"
KAGGLE_BEST_LORA   = "/kaggle/working/mcq-lora-adapter-best_v2"
KAGGLE_BEST_MERGED = "/kaggle/working/mcq-qwen-merged-best_v2"

# ── Copy best LoRA adapter files (not checkpoint sub-dirs) ────────────────────
os.makedirs(KAGGLE_BEST_LORA, exist_ok=True)
for fname in os.listdir(BEST_CKPT):
    src = os.path.join(BEST_CKPT, fname)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(KAGGLE_BEST_LORA, fname))
print(f"✓ Best LoRA adapter copied → {KAGGLE_BEST_LORA}")

# ── Copy merged model ─────────────────────────────────────────────────────────
if os.path.exists(KAGGLE_BEST_MERGED):
    shutil.rmtree(KAGGLE_BEST_MERGED)
shutil.copytree(KAGGLE_MERGED_DIR, KAGGLE_BEST_MERGED)
print(f"✓ Merged model copied      → {KAGGLE_BEST_MERGED}")

# ── Verify ────────────────────────────────────────────────────────────────────
merged_files = os.listdir(KAGGLE_BEST_MERGED)
assert any(f.endswith(".safetensors") for f in merged_files), "Merged model missing safetensors!"
assert "config.json" in merged_files, "config.json missing."
print("\n✓ Output folders verified.")
print(f"  LoRA adapter  : {KAGGLE_BEST_LORA}")
print(f"  Merged model  : {KAGGLE_BEST_MERGED}")
print("\n  Next: push both to HF Hub in §11.11.")


✓ Best LoRA adapter copied → /kaggle/working/mcq-lora-adapter-best_v2
✓ Merged model copied      → /kaggle/working/mcq-qwen-merged-best_v2

✓ Output folders verified.
  LoRA adapter  : /kaggle/working/mcq-lora-adapter-best_v2
  Merged model  : /kaggle/working/mcq-qwen-merged-best_v2

  Next: push both to HF Hub in §11.11.


### 11.11  Push Final Models to Hugging Face Hub

In [105]:
# import os

# readme_path = os.path.join(BEST_CKPT, "README.md")

# with open(readme_path, "r") as f:
#     content = f.read()

# # replace local path with HF model id
# content = content.replace(
#     "/kaggle/working/qwen_model",
#     "Qwen/Qwen2.5-1.5B-Instruct"
# )

# with open(readme_path, "w") as f:
#     f.write(content)

# print("✓ README fixed")

✓ README fixed


In [106]:
# ── Push best LoRA adapter to HF Hub ─────────────────────────────────────────
from huggingface_hub import HfApi
api = HfApi()

print(f"Uploading best LoRA adapter → {LORA_REPO_ID} ...")
api.upload_folder(
    folder_path=BEST_CKPT,
    repo_id=LORA_REPO_ID,
    repo_type="model",
)
print(f"✓ LoRA adapter pushed → https://huggingface.co/{LORA_REPO_ID}")


Uploading best LoRA adapter → Aya-khaled/mcq-bloom-qwen-lora_v2 ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ LoRA adapter pushed → https://huggingface.co/Aya-khaled/mcq-bloom-qwen-lora_v2


In [107]:
# ── Push final merged model (+ tokenizer) to HF Hub ──────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

KAGGLE_MERGED_DIR = "/kaggle/working/mcq-qwen-merged_v2"

tokenizer = AutoTokenizer.from_pretrained(KAGGLE_MERGED_DIR)
model     = AutoModelForCausalLM.from_pretrained(
    KAGGLE_MERGED_DIR, torch_dtype=torch.bfloat16, device_map="cpu"
)

print(f"Pushing merged model → {MERGED_REPO_ID} ...")
model.push_to_hub(MERGED_REPO_ID, private=True, token=HF_TOKEN)
tokenizer.push_to_hub(MERGED_REPO_ID, private=True, token=HF_TOKEN)
print(f"✓ Merged model pushed → https://huggingface.co/{MERGED_REPO_ID}")

del model
import gc; gc.collect()
torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Pushing merged model → Aya-khaled/mcq-bloom-qwen-merged_v2 ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Merged model pushed → https://huggingface.co/Aya-khaled/mcq-bloom-qwen-merged_v2


### 11.12  Storage Cleanup

Removes temporary files from Kaggle's local disk (`/kaggle/working`).
Run **after** all HF uploads and Drive saves are confirmed.

**What gets deleted:**
- All local checkpoints except BEST_CKPT (already cleaned in 11.8)
- Local merged model (already on HF + Drive)
- Kaggle disk: only the LoRA adapter dir and final merged dir may remain if needed

**What is preserved:**
- HF Hub: all checkpoints + best LoRA + merged model
- Drive: best LoRA + merged model


In [108]:
import shutil, os, glob

# ── Remove local merged model (already on HF + Drive) ────────────────────────
KAGGLE_MERGED_DIR = "/kaggle/working/mcq-qwen-merged_v2"
if os.path.exists(KAGGLE_MERGED_DIR):
    shutil.rmtree(KAGGLE_MERGED_DIR)
    print(f"✓ Deleted local merged model: {KAGGLE_MERGED_DIR}")

# ── Remove any remaining local checkpoints ────────────────────────────────────
KAGGLE_LORA_DIR = "/kaggle/working/mcq-lora-adapter_v2"
for ckpt in glob.glob(os.path.join(KAGGLE_LORA_DIR, "checkpoint-*")):
    shutil.rmtree(ckpt, ignore_errors=True)
    print(f"✓ Deleted local checkpoint: {ckpt}")

# ── Report remaining Kaggle disk usage ────────────────────────────────────────
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nKaggle /kaggle/working disk usage:")
print(f"  Used: {used // (1024**3):.1f} GB / {total // (1024**3):.1f} GB  ({free // (1024**3):.1f} GB free)")


✓ Deleted local merged model: /kaggle/working/mcq-qwen-merged_v2

Kaggle /kaggle/working disk usage:
  Used: 8.0 GB / 19.0 GB  (10.0 GB free)


### 11.13  Plot Training Curves

In [113]:
import json, os
import matplotlib.pyplot as plt

KAGGLE_LORA_DIR = "/kaggle/working/mcq-lora-adapter_v2"
trainer_state_path = os.path.join(KAGGLE_LORA_DIR, "trainer_state.json")

if not os.path.exists(trainer_state_path):
    print("trainer_state.json not found — run training first.")
else:
    with open(trainer_state_path) as f:
        state = json.load(f)

    train_steps, train_losses = [], []
    eval_steps,  eval_losses  = [], []

    for entry in state.get("log_history", []):
        step = entry.get("step", 0)
        if "loss" in entry:
            train_steps.append(step)
            train_losses.append(entry["loss"])
        if "eval_loss" in entry:
            eval_steps.append(step)
            eval_losses.append(entry["eval_loss"])

    fig, ax = plt.subplots(figsize=(12, 5))
    if train_losses:
        ax.plot(train_steps, train_losses, label="Train Loss", color="#2563EB", lw=2)
    if eval_losses:
        ax.plot(eval_steps, eval_losses, label="Eval Loss",  color="#DC2626", lw=2, marker="o", ms=5)
        best_idx = eval_losses.index(min(eval_losses))
        ax.axvline(eval_steps[best_idx], color="#16A34A", ls="--", lw=1.5,
                   label=f"Best Eval (step {eval_steps[best_idx]}, loss={min(eval_losses):.4f})")

    ax.set_xlabel("Training Step")
    ax.set_ylabel("Loss")
    ax.set_title("MCQ Qwen2.5 LoRA — Training & Eval Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()
    print("✓ Training curves saved → /kaggle/working/training_curves.png")


trainer_state.json not found — run training first.


### 11.14  Load the Merged Model for Inference

In [114]:
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# # Load from the clean best-model output folder
# KAGGLE_BEST_MERGED = "/kaggle/working/mcq-qwen-merged-best"
# KAGGLE_MERGED_DIR  = "/kaggle/working/mcq-qwen-merged"

# model_path = (
#     KAGGLE_BEST_MERGED if os.path.exists(KAGGLE_BEST_MERGED)
#     else KAGGLE_MERGED_DIR
# )
# print(f"Loading merged model from: {model_path}")

# tokenizer = AutoTokenizer.from_pretrained(model_path)
# model     = AutoModelForCausalLM.from_pretrained(
#     model_path, torch_dtype=torch.bfloat16, device_map="auto"
# )
# model.eval()
# print(f"✓ Merged model loaded on {DEVICE}")


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HF_MODEL_ID = "Aya-khaled/mcq-bloom-qwen-merged_v2"  # <-- change to your real username

print(f"Loading model from Hugging Face: {HF_MODEL_ID}")

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)

model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)

model.eval()
print(f"✓ Model loaded on {DEVICE}")

Loading model from Hugging Face: Aya-khaled/mcq-bloom-qwen-merged_v2


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

✓ Model loaded on cuda


### 11.15  Inference Helper

In [115]:
import json as _json

def generate_mcq(messages: list, max_new_tokens: int = 512) -> dict:
    """
    Run MCQ inference with the merged fine-tuned model.

    Parameters
    ----------
    messages : list[dict]  [{role, content}, ...]  (system + user turns)
    max_new_tokens : int

    Returns
    -------
    dict  parsed MCQ {question, options, answer, explanation}
          or raw str if JSON parsing fails.
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    new_ids = out_ids[0][inputs.input_ids.shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True)

    try:
        clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        return _json.loads(clean)
    except _json.JSONDecodeError:
        print("[WARN] Response is not valid JSON — returning raw string.")
        return raw

print("✓ generate_mcq() defined.")


✓ generate_mcq() defined.


### 11.18  Pre-Training Checklist (Kaggle Edition)

Go through every item before launching 11.6. Fix any failure before training.

**Kaggle Environment**
- [ ] Running on Kaggle (NOT Colab)
- [ ] Dataset `ayakhaled51/mcqs-files` added via Notebook Settings → Add Data
- [ ] Input files visible at `/kaggle/input/datasets/ayakhaled51/mcqs-files/`
- [ ] `HF_TOKEN` set in Kaggle Secrets → Settings → Secrets
- [ ] `WANDB_API_KEY` set in Kaggle Secrets (or `WANDB_MODE=disabled`)
- [ ] `HF_USERNAME` set in Kaggle Secrets
- [ ] GPU enabled in Kaggle notebook settings (T4 × 2 recommended)
- [ ] `torch.cuda.is_available()` returns `True`
- [ ] Internet enabled in Kaggle (Settings → Internet → On)

**Data**
- [ ] `sft_metadata_conditioned.jsonl` exists and is non-empty (11.1)
- [ ] `CUTOFF_LEN` set automatically from P95 — not hardcoded (11.1)
- [ ] Bloom distribution: no level is 0% in any split (11.1 / 11.2)
- [ ] Answer distribution: no key exceeds 40% (11.1)
- [ ] `mcq_sft_train.jsonl`, `mcq_sft_val.jsonl` written (11.2)
- [ ] Split total equals full record count — assertion passed (11.2)

**LLaMA-Factory**
- [ ] `git clone` done, `llamafactory-cli version` printed (11.0)
- [ ] `dataset_info.json` updated with all three splits (11.3)
- [ ] Schema smoke-test printed `SCHEMA OK` (11.3)
- [ ] YAML `output_dir` is `/kaggle/working/mcq-lora-adapter` (NOT Drive)
- [ ] YAML `push_to_hub: true` + `hub_strategy: all_checkpoints` (auto-push checkpoints to HF)
- [ ] YAML `load_best_model_at_end: true` + `metric_for_best_model: eval_loss`
- [ ] YAML `bf16: true` — never change to `fp16` for Qwen2.5
- [ ] YAML `packing: false` — required for multi-turn ShareGPT records
- [ ] YAML `lora_alpha: 128` = 2 × `lora_rank: 64`

**Storage**
- [ ] Checkpoints go to HF Hub automatically (hub_strategy: all_checkpoints)
- [ ] No checkpoints saved to Drive or permanently on Kaggle
- [ ] Best LoRA + merged model copied to `/kaggle/working/*-best/` folders (11.10)
- [ ] Best LoRA + merged model → HF Hub (11.11)
- [ ] Local cleanup done (11.12)

**Post-training targets**
- [ ] Best checkpoint auto-selected by eval_loss (trainer_state.json checked in 11.7)
- [ ] Merged model contains `.safetensors` + `config.json` (11.9)
- [ ] Training curves plotted (11.13)


---

## 12. LLM-as-a-Judge Evaluation Framework

This section evaluates three model stages on the **same** inputs using a reference-free LLM-as-a-Judge approach:

| Stage | Model | Description |
|---|---|---|
| **Stage 1 — Teacher** | Gemini flash-2.5-lite | MCQs from `teacher_outputs.jsonl` — the quality ceiling |
| **Stage 2 — Student Pre-FT** | Base Qwen2.5-1.5B-Instruct | MCQs generated **before** fine-tuning |
| **Stage 3 — Student Post-FT** | Merged Qwen (LoRA) | MCQs generated **after** fine-tuning |

**Inputs per MCQ (no reference MCQs required):**
- Generated MCQ (question, options A–D, correct answer, explanation)
- MCQ metadata: Bloom level, chunk_type, concept / learning objective
- Source chunk text (ground truth for factual grounding)
- Chunk metadata: topic / domain from `selected_subset_v3.jsonl`

**Six evaluation dimensions (1–5 Likert each):**

| ID | Dimension | Weight |
|---|---|---|
| D1 | Factual Grounding | 25% |
| D2 | Question Clarity | 10% |
| D3 | Correct Answer Validity | 20% |
| D4 | Distractor Plausibility | 20% |
| D5 | Bloom Taxonomy Alignment | 15% |
| D6 | Topic & Learning Objective Alignment | 10% |

`CMQS = 0.25×D1 + 0.10×D2 + 0.20×D3 + 0.20×D4 + 0.15×D5 + 0.10×D6`

> **Resumable:** every stage resumes from the last completed sample if interrupted. No sample is re-evaluated.


### 12.0  Evaluation Paths & Configuration

In [150]:
# # ─── 12.0  Evaluation Paths & Configuration ──────────────────────────────────
# import os
# from os.path import join

# # ── Re-establish base dirs if running standalone ──────────────────────────────
# if "KAGGLE_WORKING" not in dir():
#     KAGGLE_WORKING = "/kaggle/working"
# if "INPUT_DIR" not in dir():
#     INPUT_DIR = ""
# if "OUTPUT_DIR" not in dir():
#     OUTPUT_DIR = join(KAGGLE_WORKING, "GP_Quiz-Generation_Data")

# EVAL_DIR = join(KAGGLE_WORKING, "evaluation")
# os.makedirs(EVAL_DIR, exist_ok=True)

# # ── Input files ───────────────────────────────────────────────────────────────
# TEACHER_OUTPUT_PATH = join(INPUT_DIR, "teacher_outputs.jsonl")
# CHUNKS_PATH = join(INPUT_DIR, "selected_subset_v4.jsonl")

# # ── Raw model output files (generated by this section) ───────────────────────
# STUDENT_PRE_OUTPUT_PATH  = join(EVAL_DIR, "student_pre_ft_outputs.jsonl")
# STUDENT_POST_OUTPUT_PATH = join(EVAL_DIR, "student_post_ft_outputs.jsonl")

# # ── Judge score files (one JSON line per evaluated MCQ) ──────────────────────
# TEACHER_SCORES_PATH      = join(EVAL_DIR, "teacher_judge_scores.jsonl")
# STUDENT_PRE_SCORES_PATH  = join(EVAL_DIR, "student_pre_ft_judge_scores.jsonl")
# STUDENT_POST_SCORES_PATH = join(EVAL_DIR, "student_post_ft_judge_scores.jsonl")

# # ── Final comparison results ──────────────────────────────────────────────────
# COMPARISON_PATH = join(EVAL_DIR, "final_comparison_results.json")

# # ══════════════════════════════════════════════════════════════════════════════
# # ── OpenRouter Configuration ─────────────────────────────────────────────────
# # ══════════════════════════════════════════════════════════════════════════════
# #
# # 1) Set your Kaggle Secret:  OPENROUTER_API_KEY
# #    (Go to Notebook → Add-ons → Secrets → Add → name: OPENROUTER_API_KEY)
# #
# # 2) Pick your judge model from the dict below, or set JUDGE_MODEL directly.
# #    All models use the OpenRouter API (https://openrouter.ai/docs).
# # ──────────────────────────────────────────────────────────────────────────────

# OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# # ── Available judge models (easy to switch) ──────────────────────────────────
# # Uncomment ONE line below, or set JUDGE_MODEL to any OpenRouter model slug.
# # Pricing and context info: https://openrouter.ai/models
# #
# # --- Free / Very Cheap ---
# # JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct:free"       # Free, 70B
# # JUDGE_MODEL = "deepseek/deepseek-chat-v3-0324:free"           # Free, DeepSeek V3
# # JUDGE_MODEL = "qwen/qwen3-235b-a22b:free"                    # Free, Qwen3 MoE
# # JUDGE_MODEL = "google/gemini-2.0-flash-001"                   # Very cheap
# #
# # --- Mid-Range ---
# # JUDGE_MODEL = "anthropic/claude-sonnet-4"                     # Great quality
# # JUDGE_MODEL = "meta-llama/llama-4-maverick"                   # Llama 4
# #
# # --- Premium ---
# # JUDGE_MODEL = "openai/gpt-4.1"                               # GPT-4.1
# # JUDGE_MODEL = "anthropic/claude-opus-4"                       # Claude Opus 4
# # JUDGE_MODEL = "google/gemini-2.5-pro-preview-06-05"           # Gemini 2.5 Pro

# JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct:free" 

# # ── Load OpenRouter API key from Kaggle Secrets ──────────────────────────────
# try:
#     from kaggle_secrets import UserSecretsClient
#     _secrets = UserSecretsClient()
#     OPENROUTER_API_KEY = _secrets.get_secret("OPENROUTER_API_KEY")
#     os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
#     print("✓ OPENROUTER_API_KEY loaded from Kaggle Secrets.")
# except Exception as _e:
#     OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
#     print(f"  OPENROUTER_API_KEY fallback to env var: {'found' if OPENROUTER_API_KEY else 'MISSING'}  ({_e})")

# if not OPENROUTER_API_KEY:
#     print("\n⚠️  WARNING: OPENROUTER_API_KEY is not set!")
#     print("   → Go to Notebook → Add-ons → Secrets → Add a new secret")
#     print("   → Name: OPENROUTER_API_KEY")
#     print("   → Value: your key from https://openrouter.ai/settings/keys")

# # ── Evaluation sample size ────────────────────────────────────────────────────
# EVAL_SAMPLE_N = 100   # <-- change to None for full evaluation

# print()
# print("Evaluation directory :", EVAL_DIR)
# print("Teacher outputs      :", TEACHER_OUTPUT_PATH)
# print("Chunks path          :", CHUNKS_PATH)
# print()
# print("Output files")
# print("  Student Pre-FT raw     :", STUDENT_PRE_OUTPUT_PATH)
# print("  Student Post-FT raw    :", STUDENT_POST_OUTPUT_PATH)
# print("  Teacher scores         :", TEACHER_SCORES_PATH)
# print("  Student Pre-FT scores  :", STUDENT_PRE_SCORES_PATH)
# print("  Student Post-FT scores :", STUDENT_POST_SCORES_PATH)
# print("  Final comparison       :", COMPARISON_PATH)
# print()
# print(f"Judge model   : {JUDGE_MODEL}")
# print(f"Judge backend : OpenRouter ({OPENROUTER_BASE_URL})")
# print(f"Eval sample N : {EVAL_SAMPLE_N if EVAL_SAMPLE_N else 'ALL'}")


✓ OPENROUTER_API_KEY loaded from Kaggle Secrets.

Evaluation directory : /kaggle/working/evaluation
Teacher outputs      : /kaggle/input/datasets/ayakhaled51/mcqs-files/teacher_outputs.jsonl
Chunks path          : /kaggle/input/datasets/ayakhaled51/mcqs-files/selected_subset_v4.jsonl

Output files
  Student Pre-FT raw     : /kaggle/working/evaluation/student_pre_ft_outputs.jsonl
  Student Post-FT raw    : /kaggle/working/evaluation/student_post_ft_outputs.jsonl
  Teacher scores         : /kaggle/working/evaluation/teacher_judge_scores.jsonl
  Student Pre-FT scores  : /kaggle/working/evaluation/student_pre_ft_judge_scores.jsonl
  Student Post-FT scores : /kaggle/working/evaluation/student_post_ft_judge_scores.jsonl
  Final comparison       : /kaggle/working/evaluation/final_comparison_results.json

Judge model   : meta-llama/llama-3.3-70b-instruct:free
Judge backend : OpenRouter (https://openrouter.ai/api/v1)
Eval sample N : 100


In [155]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.8 MB/s eta 0:00:00a 0:00:01


In [182]:
# ─── 12.0  Evaluation Paths & Configuration ──────────────────────────────────
import os
from os.path import join

# ── Re-establish base dirs if running standalone ──────────────────────────────
if "KAGGLE_WORKING" not in dir():
    KAGGLE_WORKING = "/kaggle/working"
if "INPUT_DIR" not in dir():
    INPUT_DIR = ""
if "OUTPUT_DIR" not in dir():
    OUTPUT_DIR = join(KAGGLE_WORKING, "GP_Quiz-Generation_Data")

EVAL_DIR = join(KAGGLE_WORKING, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)

# ── Input files ───────────────────────────────────────────────────────────────
TEACHER_OUTPUT_PATH = join(INPUT_DIR, "teacher_outputs.jsonl")
CHUNKS_PATH = join(INPUT_DIR, "selected_subset_v4.jsonl")

# ── Raw model output files ────────────────────────────────────────────────────
STUDENT_PRE_OUTPUT_PATH  = join(EVAL_DIR, "student_pre_ft_outputs.jsonl")
STUDENT_POST_OUTPUT_PATH = join(EVAL_DIR, "student_post_ft_outputs.jsonl")

# ── Judge score files ─────────────────────────────────────────────────────────
TEACHER_SCORES_PATH      = join(EVAL_DIR, "teacher_judge_scores.jsonl")
STUDENT_PRE_SCORES_PATH  = join(EVAL_DIR, "student_pre_ft_judge_scores.jsonl")
STUDENT_POST_SCORES_PATH = join(EVAL_DIR, "student_post_ft_judge_scores.jsonl")

# ── Final comparison results ─────────────────────────────────────────────────
COMPARISON_PATH = join(EVAL_DIR, "final_comparison_results.json")

# ══════════════════════════════════════════════════════════════════════════════
# ── GROQ CONFIGURATION (REPLACES OPENROUTER) ─────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# Install groq if needed
# !pip install groq

from groq import Groq

# ── Load API key from Kaggle Secrets ──────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    GROQ_API_KEY = _secrets.get_secret("GROQ_API_KEY")
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("✓ GROQ_API_KEY loaded from Kaggle Secrets.")
except Exception as _e:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    print(f"  GROQ_API_KEY fallback to env var: {'found' if GROQ_API_KEY else 'MISSING'}  ({_e})")

if not GROQ_API_KEY:
    print("\n⚠️ WARNING: GROQ_API_KEY is not set!")
    print("   → Add it in Kaggle Secrets as GROQ_API_KEY")
    print("   → Get it from https://console.groq.com/keys")

# ── Initialize Groq client ────────────────────────────────────────────────────
client = Groq(api_key=GROQ_API_KEY)

# ── Judge model (Groq-supported LLaMA) ────────────────────────────────────────
JUDGE_MODEL = "llama-3.3-70b-versatile"
# Alternative: "llama3-8b-8192" (faster, lower quality)

# ── Evaluation sample size ────────────────────────────────────────────────────
EVAL_SAMPLE_N = 100   # <-- change to None for full evaluation

print()
print("Evaluation directory :", EVAL_DIR)
print("Teacher outputs      :", TEACHER_OUTPUT_PATH)
print("Chunks path          :", CHUNKS_PATH)
print()
print("Output files")
print("  Student Pre-FT raw     :", STUDENT_PRE_OUTPUT_PATH)
print("  Student Post-FT raw    :", STUDENT_POST_OUTPUT_PATH)
print("  Teacher scores         :", TEACHER_SCORES_PATH)
print("  Student Pre-FT scores  :", STUDENT_PRE_SCORES_PATH)
print("  Student Post-FT scores :", STUDENT_POST_SCORES_PATH)
print("  Final comparison       :", COMPARISON_PATH)
print()
print(f"Judge model   : {JUDGE_MODEL} (Groq)")
print(f"Judge backend : Groq API (https://console.groq.com)")
print(f"Eval sample N : {EVAL_SAMPLE_N if EVAL_SAMPLE_N else 'ALL'}")

✓ GROQ_API_KEY loaded from Kaggle Secrets.

Evaluation directory : /kaggle/working/evaluation
Teacher outputs      : /kaggle/input/datasets/ayakhaled51/mcqs-files/teacher_outputs.jsonl
Chunks path          : /kaggle/input/datasets/ayakhaled51/mcqs-files/selected_subset_v4.jsonl

Output files
  Student Pre-FT raw     : /kaggle/working/evaluation/student_pre_ft_outputs.jsonl
  Student Post-FT raw    : /kaggle/working/evaluation/student_post_ft_outputs.jsonl
  Teacher scores         : /kaggle/working/evaluation/teacher_judge_scores.jsonl
  Student Pre-FT scores  : /kaggle/working/evaluation/student_pre_ft_judge_scores.jsonl
  Student Post-FT scores : /kaggle/working/evaluation/student_post_ft_judge_scores.jsonl
  Final comparison       : /kaggle/working/evaluation/final_comparison_results.json

Judge model   : llama-3.3-70b-versatile (Groq)
Judge backend : Groq API (https://console.groq.com)
Eval sample N : 100


### 12.1  Judge Model — `gpt-oss-120b` and Free Alternatives

**Primary judge: `gpt-oss-120b`**
Available on Kaggle via the Kaggle Models hub (`kaggle/models/openai/gpt-oss`). Uses the standard OpenAI-compatible `/v1/chat/completions` API. No external API key needed when running on Kaggle.

---

**Free alternatives (ranked by recommendation for this task):**

| Model | Why suitable | How to access |
|---|---|---|
| **`meta-llama/Llama-3.3-70B-Instruct`** (⭐ top free pick) | Strongest open model for structured evaluation; excellent JSON adherence; superior to gpt-oss-120b on educational tasks in benchmarks | Kaggle Models → set `JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct"` |
| **`google/gemma-3-27b-it`** | Strong instruction following; Google-trained; good Bloom taxonomy understanding | Kaggle Models → set `JUDGE_MODEL = "google/gemma-3-27b-it"` |
| **`Qwen/Qwen2.5-72B-Instruct`** | Very strong at structured JSON output; good for educational tasks; same model family as student | Kaggle Models or HF Inference |
| **`mistralai/Mistral-Large-Instruct-2411`** | Strong reasoning; good JSON mode support | HF Inference API (free tier) |

> **Important:** Do NOT use the same model family as your student (Qwen2.5-1.5B) as the judge — small models are unreliable as judges and same-family models exhibit self-preference bias. The recommended free alternative is **Llama-3.3-70B-Instruct**.

To switch: change `JUDGE_MODEL` in cell 12.0.


### 12.2  Shared Utilities

In [183]:
# # ─── 12.2  Shared Utilities ──────────────────────────────────────────────────
# import json, os, time, re
# from typing import Dict, List, Optional, Tuple

# # ── JSON-L helpers ────────────────────────────────────────────────────────────

# def load_jsonl_safe(path: str) -> List[Dict]:
#     """Load a JSONL, skipping blank/malformed lines."""
#     if not os.path.exists(path):
#         return []
#     records = []
#     with open(path, encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if not line:
#                 continue
#             try:
#                 records.append(json.loads(line))
#             except json.JSONDecodeError:
#                 pass
#     return records


# def append_jsonl(path: str, record: Dict) -> None:
#     """Append one record to a JSONL file (atomic per-sample save)."""
#     with open(path, "a", encoding="utf-8") as f:
#         f.write(json.dumps(record, ensure_ascii=False) + "\n")


# def load_done_ids(scores_path: str) -> set:
#     """Return the set of chunk_id+slot_index keys already scored (for resume)."""
#     done = set()
#     for r in load_jsonl_safe(scores_path):
#         key = f"{r.get('chunk_id','')}_{r.get('slot_index','')}"
#         done.add(key)
#     return done


# def record_key(rec: Dict) -> str:
#     return f"{rec.get('chunk_id','')}_{rec.get('slot_index','')}"

# # ── Chunk lookup ──────────────────────────────────────────────────────────────

# def build_chunk_lookup(chunks_path: str) -> Dict[str, Dict]:
#     """Build chunk_id → chunk_record lookup from selected_subset_v3.jsonl."""
#     lookup = {}
#     for r in load_jsonl_safe(chunks_path):
#         cid = r.get("chunk_id")
#         if cid:
#             lookup[cid] = r
#     print(f"Chunk lookup built: {len(lookup):,} chunks from {os.path.basename(chunks_path)}")
#     return lookup

# # ══════════════════════════════════════════════════════════════════════════════
# # ── OpenRouter LLM Judge Caller ──────────────────────────────────────────────
# # ══════════════════════════════════════════════════════════════════════════════

# def call_judge(
#     prompt_messages: List[Dict],
#     model: str = None,
#     max_retries: int = 3,
#     retry_delay: float = 5.0,
# ) -> Optional[str]:
#     """
#     Call the LLM judge via OpenRouter's OpenAI-compatible API.

#     Uses the OPENROUTER_API_KEY Kaggle secret and OPENROUTER_BASE_URL.
#     The model can be any OpenRouter model slug (e.g. 'google/gemini-2.5-flash-preview-05-20').

#     Parameters
#     ----------
#     prompt_messages : list of {role, content} dicts (system + user)
#     model           : OpenRouter model slug. Defaults to global JUDGE_MODEL.
#     max_retries     : number of retry attempts on failure
#     retry_delay     : seconds between retries (with exponential back-off)

#     Returns
#     -------
#     Raw text content of the response, or None on total failure.
#     """
#     try:
#         import openai
#     except ImportError:
#         raise ImportError("Run: pip install -q openai")

#     if model is None:
#         model = JUDGE_MODEL

#     # Build the OpenRouter client
#     api_key = os.environ.get("OPENROUTER_API_KEY", "")
#     if not api_key:
#         raise ValueError(
#             "OPENROUTER_API_KEY not found. "
#             "Add it as a Kaggle Secret: Notebook → Add-ons → Secrets → "
#             "Name: OPENROUTER_API_KEY, Value: your key from https://openrouter.ai/settings/keys"
#         )

#     client = openai.OpenAI(
#         base_url=OPENROUTER_BASE_URL,
#         api_key=api_key,
#         default_headers={
#             "HTTP-Referer": "https://www.kaggle.com",
#             "X-Title":      "MCQ-LLM-Judge-Evaluation",
#         },
#     )

#     for attempt in range(1, max_retries + 1):
#         try:
#             resp = client.chat.completions.create(
#                 model=model,
#                 messages=prompt_messages,
#                 temperature=0.1,
#                 max_tokens=1024,
#                 # Note: response_format={"type":"json_object"} may not be
#                 # supported by all OpenRouter models. We handle JSON parsing
#                 # in parse_judge_json() with fallback stripping.
#             )
#             content = resp.choices[0].message.content
#             if content:
#                 return content
#             print(f"  [Judge] Attempt {attempt}: empty response, retrying...")
#         except Exception as exc:
#             print(f"  [Judge] Attempt {attempt}/{max_retries} failed: {exc}")
#             if attempt < max_retries:
#                 wait = retry_delay * (2 ** (attempt - 1))  # exponential back-off
#                 time.sleep(wait)
#     return None


# def parse_judge_json(raw: Optional[str]) -> Optional[Dict]:
#     """Parse judge response JSON, stripping any markdown fences."""
#     if not raw:
#         return None
#     try:
#         clean = raw.strip()
#         # Strip markdown code fences if present
#         clean = re.sub(r"^```json\s*", "", clean)
#         clean = re.sub(r"^```\s*", "", clean)
#         clean = re.sub(r"\s*```$", "", clean)
#         return json.loads(clean)
#     except json.JSONDecodeError:
#         # Try to find JSON object in the response
#         match = re.search(r'\{.*\}', raw, re.DOTALL)
#         if match:
#             try:
#                 return json.loads(match.group())
#             except json.JSONDecodeError:
#                 pass
#         return None


# def compute_cmqs(scores: Dict) -> float:
#     """Compute weighted CMQS from dimension scores dict."""
#     weights = {
#         "D1_factual_grounding":      0.25,
#         "D2_question_clarity":       0.10,
#         "D3_correct_answer_validity":0.20,
#         "D4_distractor_plausibility":0.20,
#         "D5_bloom_alignment":        0.15,
#         "D6_topic_lo_alignment":     0.10,
#     }
#     total, weight_sum = 0.0, 0.0
#     for key, w in weights.items():
#         val = scores.get(key, {})
#         s = val.get("score") if isinstance(val, dict) else val
#         if s is not None:
#             total += float(s) * w
#             weight_sum += w
#     if weight_sum == 0:
#         return 0.0
#     return round(total / weight_sum, 4)


# print("✓ Shared utilities defined (OpenRouter backend).")
# print(f"  Judge model  : {JUDGE_MODEL}")
# print(f"  Base URL     : {OPENROUTER_BASE_URL}")




# ─── 12.2  Shared Utilities ──────────────────────────────────────────────────
import json, os, time, re
from typing import Dict, List, Optional, Tuple

# ── JSON-L helpers ────────────────────────────────────────────────────────────

def load_jsonl_safe(path: str) -> List[Dict]:
    if not os.path.exists(path):
        return []
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return records


def append_jsonl(path: str, record: Dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_done_ids(scores_path: str) -> set:
    done = set()
    for r in load_jsonl_safe(scores_path):
        key = f"{r.get('chunk_id','')}_{r.get('slot_index','')}"
        done.add(key)
    return done


def record_key(rec: Dict) -> str:
    return f"{rec.get('chunk_id','')}_{rec.get('slot_index','')}"


# ── Chunk lookup ──────────────────────────────────────────────────────────────

def build_chunk_lookup(chunks_path: str) -> Dict[str, Dict]:
    lookup = {}
    for r in load_jsonl_safe(chunks_path):
        cid = r.get("chunk_id")
        if cid:
            lookup[cid] = r
    print(f"Chunk lookup built: {len(lookup):,} chunks from {os.path.basename(chunks_path)}")
    return lookup


# ══════════════════════════════════════════════════════════════════════════════
# ── GROQ LLM Judge Caller (REPLACES OPENROUTER) ──────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

from groq import Groq

api_key = os.environ.get("GROQ_API_KEY", "")

if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("GROQ_API_KEY")
        os.environ["GROQ_API_KEY"] = api_key
    except Exception as e:
        print("⚠️ GROQ_API_KEY not found!")

client = Groq(api_key=api_key)

JUDGE_MODEL = "llama-3.3-70b-versatile"


def call_judge(
    prompt_messages: List[Dict],
    model: str = None,
    max_retries: int = 3,
    retry_delay: float = 5.0,
) -> Optional[str]:

    if model is None:
        model = JUDGE_MODEL

    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=prompt_messages,
                temperature=0.1,
                max_tokens=1024,
            )

            content = resp.choices[0].message.content
            if content:
                return content

            print(f"  [Judge] Attempt {attempt}: empty response, retrying...")

        except Exception as exc:
            print(f"  [Judge] Attempt {attempt}/{max_retries} failed: {exc}")
            if attempt < max_retries:
                wait = retry_delay * (2 ** (attempt - 1))
                time.sleep(wait)

    return None


def parse_judge_json(raw: Optional[str]) -> Optional[Dict]:
    if not raw:
        return None
    try:
        clean = raw.strip()
        clean = re.sub(r"^```json\s*", "", clean)
        clean = re.sub(r"^```\s*", "", clean)
        clean = re.sub(r"\s*```$", "", clean)
        return json.loads(clean)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return None


def compute_cmqs(scores: Dict) -> float:
    weights = {
        "D1_factual_grounding":      0.25,
        "D2_question_clarity":       0.10,
        "D3_correct_answer_validity":0.20,
        "D4_distractor_plausibility":0.20,
        "D5_bloom_alignment":        0.15,
        "D6_topic_lo_alignment":     0.10,
    }

    total, weight_sum = 0.0, 0.0

    for key, w in weights.items():
        val = scores.get(key, {})
        s = val.get("score") if isinstance(val, dict) else val
        if s is not None:
            total += float(s) * w
            weight_sum += w

    if weight_sum == 0:
        return 0.0

    return round(total / weight_sum, 4)


print("✓ Shared utilities defined (GROQ backend).")
print(f"  Judge model : {JUDGE_MODEL}")
print("  Backend     : Groq API (https://console.groq.com)")

✓ Shared utilities defined (GROQ backend).
  Judge model : llama-3.3-70b-versatile
  Backend     : Groq API (https://console.groq.com)


### 12.3  LLM Judge Prompt Builder

In [171]:
# ─── 12.3  LLM Judge Prompt Builder ─────────────────────────────────────────

JUDGE_SYSTEM_PROMPT = """You are an expert educational assessment evaluator with deep knowledge of Bloom's Taxonomy and MCQ item analysis.
You will receive a multiple-choice question (MCQ), its metadata, and the source text chunk it was generated from.
The source chunk is the ONLY source of truth for factual correctness.
Evaluate the MCQ on exactly six dimensions using a strict 1-5 integer scale.
Output ONLY valid JSON. No preamble, explanation, or text outside the JSON object.""".strip()

JUDGE_USER_TEMPLATE = """=== SOURCE CHUNK ===
{source_text}

=== CHUNK METADATA ===
Chunk Type: {chunk_type}
Concept / Topic: {concept}

=== GENERATED MCQ ===
Question: {question}
A) {opt_a}
B) {opt_b}
C) {opt_c}
D) {opt_d}
Correct Answer: {correct_answer}

=== MCQ METADATA ===
Bloom Taxonomy Level: {bloom_level}
Focus Concept: {focus_concept}

=== EVALUATION TASK ===
Score each dimension from 1 (worst) to 5 (best) using ONLY the provided content above.

D1 - Factual Grounding (weight 25%):
  Is every statement in the question stem AND all four options directly supported by the source chunk?
  5=fully grounded, 4=one minor implicit inference, 3=one ungrounded claim, 2=multiple hallucinations, 1=largely hallucinated

D2 - Question Clarity (weight 10%):
  Is the question grammatically correct, unambiguous, and self-contained?
  5=perfect, 4=minor verbosity/grammar, 3=some ambiguity, 2=notably unclear, 1=incomprehensible

D3 - Correct Answer Validity (weight 20%):
  Is the designated correct answer factually correct per the source chunk AND unambiguously better than all distractors?
  5=unambiguously correct, 4=correct but slightly imprecise, 3=correct but a distractor is close, 2=partially correct, 1=wrong

D4 - Distractor Plausibility (weight 20%):
  Are ALL three distractors plausible to a student without mastery? Clearly wrong only to an expert?
  5=all highly plausible/misconception-based, 4=mostly plausible, 3=mixed quality, 2=mostly implausible, 1=trivially wrong/irrelevant

D5 - Bloom Alignment (weight 15%):
  Does the cognitive demand required to answer match the assigned Bloom level ({bloom_level})?
  Levels: remember<understand<apply<analyze<evaluate<create
  5=exact match, 4=one adjacent level, 3=adjacent level, 2=two+ levels off, 1=completely wrong

D6 - Topic & Learning Objective Alignment (weight 10%):
  Does the question genuinely test the focus concept and match the chunk's topic?
  5=fully aligned, 4=mostly aligned, 3=loosely related, 2=tangentially related, 1=off-topic

Return EXACTLY this JSON structure (integer scores, one-sentence reasons):
{{
  "D1_factual_grounding":       {{"score": <1-5>, "reason": "<one sentence>"}},
  "D2_question_clarity":        {{"score": <1-5>, "reason": "<one sentence>"}},
  "D3_correct_answer_validity": {{"score": <1-5>, "reason": "<one sentence>"}},
  "D4_distractor_plausibility": {{"score": <1-5>, "reason": "<one sentence>"}},
  "D5_bloom_alignment":         {{"score": <1-5>, "reason": "<one sentence>"}},
  "D6_topic_lo_alignment":      {{"score": <1-5>, "reason": "<one sentence>"}}
}}""".strip()


def build_judge_messages(mcq_record: Dict, chunk_record: Dict) -> List[Dict]:
    """
    Build the [system, user] message list for the LLM judge.

    Parameters
    ----------
    mcq_record   : one record from teacher / student output JSONL
                   Must have: chunk_id, text, bloom_level, chunk_type,
                              concept, mcq {question, options, answer}
    chunk_record : matching record from selected_subset_v3.jsonl
                   Must have: text, metadata {bloom_level, chunk_type, concepts}
    """
    # ── Extract MCQ fields ─────────────────────────────────────────────────────
    mcq      = mcq_record.get("mcq", {})
    question = mcq.get("question", "").strip()
    options  = mcq.get("options", {})
    answer   = mcq.get("answer", "").strip().upper()
    bloom    = mcq_record.get("bloom_level", "").strip()
    concept  = mcq_record.get("concept", "").strip()
    chunk_t  = mcq_record.get("chunk_type", "").strip()

    # Source text: prefer chunk_record text (canonical), fall back to mcq_record text
    source_text = chunk_record.get("text", mcq_record.get("text", "")).strip()

    # Chunk metadata for domain / concept context
    chunk_meta   = chunk_record.get("metadata", {})
    chunk_type_c = chunk_meta.get("chunk_type", chunk_t) or chunk_t
    concepts_c   = chunk_meta.get("concepts", [concept]) or [concept]
    concept_str  = ", ".join(concepts_c) if concepts_c else concept

    user_content = JUDGE_USER_TEMPLATE.format(
        source_text    = source_text,
        chunk_type     = chunk_type_c,
        concept        = concept_str,
        question       = question,
        opt_a          = options.get("A", ""),
        opt_b          = options.get("B", ""),
        opt_c          = options.get("C", ""),
        opt_d          = options.get("D", ""),
        correct_answer = answer,
        bloom_level    = bloom,
        focus_concept  = concept,
    )

    return [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]


print("✓ Judge prompt builder defined.")
print(f"  System prompt : {len(JUDGE_SYSTEM_PROMPT):,} chars")
print(f"  User template : {len(JUDGE_USER_TEMPLATE):,} chars")


✓ Judge prompt builder defined.
  System prompt : 457 chars
  User template : 2,555 chars


### 12.4  Core Evaluation Runner (Resumable)

In [172]:
# ─── 12.4  Core Evaluation Runner (resumable) ────────────────────────────────
from tqdm.auto import tqdm

def evaluate_stage(
    records:        List[Dict],
    chunk_lookup:   Dict[str, Dict],
    scores_path:    str,
    stage_label:    str,
) -> List[Dict]:
    """
    Run LLM-as-a-Judge evaluation on a list of MCQ records.
    Fully resumable: already-scored records are skipped.

    Parameters
    ----------
    records       : list of MCQ records (teacher or student output JSONL)
    chunk_lookup  : chunk_id -> chunk_record from selected_subset_v3.jsonl
    scores_path   : path to JSONL file where scores are appended
    stage_label   : "teacher" | "student_pre_ft" | "student_post_ft"

    Returns
    -------
    list of all score records (previously saved + newly computed)
    """
    done_ids   = load_done_ids(scores_path)
    all_scores = load_jsonl_safe(scores_path)

    to_score = [r for r in records if record_key(r) not in done_ids]
    print(f"\n{'='*60}")
    print(f"Stage        : {stage_label}")
    print(f"Total records: {len(records):,}")
    print(f"Already done : {len(done_ids):,}")
    print(f"To evaluate  : {len(to_score):,}")
    print(f"Scores file  : {scores_path}")
    print(f"{'='*60}")

    if not to_score:
        print("✓ All records already evaluated. Nothing to do.")
        return all_scores

    skipped, failed = 0, 0

    for rec in tqdm(to_score, desc=f"Judging [{stage_label}]"):
        chunk_id    = rec.get("chunk_id", "")
        slot_index  = rec.get("slot_index", 0)
        chunk_rec   = chunk_lookup.get(chunk_id, {})

        # Skip records with no matching chunk (can't evaluate grounding)
        if not chunk_rec:
            skipped += 1
            score_rec = {
                "chunk_id":    chunk_id,
                "slot_index":  slot_index,
                "stage":       stage_label,
                "status":      "skipped_no_chunk",
                "CMQS":        None,
                "scores":      None,
            }
            append_jsonl(scores_path, score_rec)
            all_scores.append(score_rec)
            continue

        # Validate MCQ structure
        mcq = rec.get("mcq", {})
        if not mcq.get("question") or not mcq.get("options") or not mcq.get("answer"):
            skipped += 1
            score_rec = {
                "chunk_id":   chunk_id,
                "slot_index": slot_index,
                "stage":      stage_label,
                "status":     "skipped_invalid_mcq",
                "CMQS":       None,
                "scores":     None,
            }
            append_jsonl(scores_path, score_rec)
            all_scores.append(score_rec)
            continue

        # Build judge prompt and call judge
        messages = build_judge_messages(rec, chunk_rec)
        raw_resp = call_judge(messages, model=JUDGE_MODEL)
        parsed   = parse_judge_json(raw_resp)

        if parsed is None:
            failed += 1
            score_rec = {
                "chunk_id":    chunk_id,
                "slot_index":  slot_index,
                "stage":       stage_label,
                "status":      "judge_parse_failed",
                "raw_response":raw_resp,
                "CMQS":        None,
                "scores":      None,
            }
            append_jsonl(scores_path, score_rec)
            all_scores.append(score_rec)
            continue

        # Validate all 6 dimension scores are present and in [1,5]
        dim_keys = [
            "D1_factual_grounding", "D2_question_clarity",
            "D3_correct_answer_validity", "D4_distractor_plausibility",
            "D5_bloom_alignment", "D6_topic_lo_alignment",
        ]
        dim_scores_valid = all(
            isinstance(parsed.get(k, {}).get("score"), (int, float))
            and 1 <= parsed[k]["score"] <= 5
            for k in dim_keys
            if k in parsed
        )
        if not dim_scores_valid or not all(k in parsed for k in dim_keys):
            failed += 1
            score_rec = {
                "chunk_id":    chunk_id,
                "slot_index":  slot_index,
                "stage":       stage_label,
                "status":      "judge_schema_invalid",
                "raw_response":raw_resp,
                "CMQS":        None,
                "scores":      parsed,
            }
            append_jsonl(scores_path, score_rec)
            all_scores.append(score_rec)
            continue

        # Success — compute CMQS and save
        cmqs = compute_cmqs(parsed)
        score_rec = {
            "chunk_id":   chunk_id,
            "slot_index": slot_index,
            "bloom_level":rec.get("bloom_level", ""),
            "chunk_type": rec.get("chunk_type", ""),
            "stage":      stage_label,
            "status":     "ok",
            "CMQS":       cmqs,
            "scores": {
                k: parsed[k]
                for k in dim_keys
            },
        }
        append_jsonl(scores_path, score_rec)
        all_scores.append(score_rec)

    print(f"\n✓ Done.  Skipped={skipped}  Failed={failed}  "
          f"Scored={len(to_score)-skipped-failed}")
    return all_scores


print("✓ evaluate_stage() defined.")


✓ evaluate_stage() defined.


### 12.5  Build Shared Evaluation Sample

In [159]:
# ─── 12.5  Build Shared Evaluation Sample ────────────────────────────────────
# We evaluate all three models on the SAME set of chunk inputs.
# The teacher JSONL is the reference for which chunks to evaluate.
# Student models are run on the same chunk_ids.

import random
from collections import defaultdict

# ── Load teacher outputs & chunk lookup ──────────────────────────────────────
chunk_lookup = build_chunk_lookup(CHUNKS_PATH)

teacher_records_all = load_jsonl_safe(TEACHER_OUTPUT_PATH)
if not teacher_records_all:
    raise FileNotFoundError(
        f"Teacher output file not found or empty: {TEACHER_OUTPUT_PATH}\n"
        "Ensure 'teacher_outputs.jsonl' is in your Kaggle input dataset."
    )
print(f"Teacher records loaded: {len(teacher_records_all):,}")

# ── Stratified sample by Bloom level (or use all) ────────────────────────────
def stratified_sample(records: List[Dict], n: Optional[int], seed: int = 42) -> List[Dict]:
    """
    Return a stratified random sample of n records balanced by Bloom level.
    If n is None or n >= len(records), returns all records (shuffled).
    """
    if n is None or n >= len(records):
        rng = random.Random(seed)
        out = records[:]
        rng.shuffle(out)
        return out

    by_bloom = defaultdict(list)
    for r in records:
        by_bloom[r.get("bloom_level", "unknown")].append(r)

    n_levels  = len(by_bloom)
    per_level = max(1, n // n_levels)
    rng       = random.Random(seed)
    sampled   = []
    for bloom_recs in by_bloom.values():
        rng.shuffle(bloom_recs)
        sampled.extend(bloom_recs[:per_level])

    # top-up if under n due to rounding
    rng.shuffle(sampled)
    remaining = [r for r in records if r not in sampled]
    rng.shuffle(remaining)
    sampled.extend(remaining[:max(0, n - len(sampled))])
    rng.shuffle(sampled)
    return sampled[:n]

teacher_eval_records = stratified_sample(teacher_records_all, EVAL_SAMPLE_N)
eval_chunk_ids       = {r["chunk_id"] for r in teacher_eval_records}

print(f"\nEvaluation sample:")
print(f"  Teacher records : {len(teacher_eval_records):,}")
print(f"  Unique chunk_ids: {len(eval_chunk_ids):,}")

from collections import Counter
bloom_dist = Counter(r.get("bloom_level", "?") for r in teacher_eval_records)
print("  Bloom distribution:")
for lvl, cnt in sorted(bloom_dist.items()):
    print(f"    {lvl:<12}: {cnt:3d}  ({cnt/len(teacher_eval_records):.1%})")

Chunk lookup built: 555 chunks from selected_subset_v4.jsonl
Teacher records loaded: 100

Evaluation sample:
  Teacher records : 100
  Unique chunk_ids: 88
  Bloom distribution:
    analyze     :  25  (25.0%)
    apply       :  31  (31.0%)
    create      :   5  (5.0%)
    evaluate    :  13  (13.0%)
    remember    :   4  (4.0%)
    understand  :  22  (22.0%)


### 12.6  Student Model Inference — Pre-Fine-Tuning

Generates MCQs using the **base Qwen2.5-1.5B-Instruct** model (before fine-tuning).
Uses the exact same system + user prompt format as the SFT training pipeline.
Fully resumable — skips already-generated samples.


In [ ]:
# ─── 12.6  Student Inference — Pre-Fine-Tuning ───────────────────────────────
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load base (pre-FT) model ──────────────────────────────────────────────────
# base_save_dir = os.path.join(KAGGLE_WORKING, "/kaggle/working/qwen_model")
base_save_dir = "/kaggle/working/qwen_model"

if os.path.exists(base_save_dir):
    pre_ft_model_path = base_save_dir
    print(f"Loading base model from local cache: {pre_ft_model_path}")
else:
    pre_ft_model_path = "Qwen/Qwen2.5-1.5B-Instruct"
    print(f"Loading base model from HuggingFace: {pre_ft_model_path}")

pre_tokenizer = AutoTokenizer.from_pretrained(pre_ft_model_path, token=HF_TOKEN)
pre_model     = AutoModelForCausalLM.from_pretrained(
    pre_ft_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN
)
pre_model.eval()
print(f"✓ Pre-FT model loaded on {DEVICE}")


def _run_student_inference(
    model, tokenizer, messages: List[Dict], max_new_tokens: int = 512
) -> dict:
    """Run a single inference call. Returns parsed MCQ dict or raw str."""
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    new_ids = out_ids[0][inputs.input_ids.shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True)
    try:
        clean = raw.strip()
        clean = re.sub(r"^```json\s*", "", clean)
        clean = re.sub(r"^```\s*", "", clean)
        clean = re.sub(r"\s*```$", "", clean)
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"_raw": raw}


def generate_student_outputs(
    model, tokenizer,
    teacher_eval_records: List[Dict],
    chunk_lookup:         Dict[str, Dict],
    output_path:          str,
    stage_label:          str,
) -> List[Dict]:
    """
    Generate MCQ outputs for all teacher eval records using the given model.
    Uses the EXACT same SFT prompt format (SFT_SYSTEM_PROMPT + build_user_prompt_conditioned).
    Fully resumable: skips already-generated samples.
    Returns list of all output records.
    """
    # Load already-done IDs
    existing     = load_jsonl_safe(output_path)
    done_keys    = {record_key(r) for r in existing}
    all_outputs  = existing[:]
    to_generate  = [r for r in teacher_eval_records if record_key(r) not in done_keys]

    print(f"\n{'='*60}")
    print(f"Stage         : {stage_label}")
    print(f"Total records : {len(teacher_eval_records):,}")
    print(f"Already done  : {len(done_keys):,}")
    print(f"To generate   : {len(to_generate):,}")
    print(f"Output file   : {output_path}")
    print(f"{'='*60}")

    if not to_generate:
        print("✓ All outputs already generated.")
        return all_outputs

    for rec in tqdm(to_generate, desc=f"Generating [{stage_label}]"):
        chunk_id   = rec.get("chunk_id", "")
        slot_index = rec.get("slot_index", 0)
        chunk_rec  = chunk_lookup.get(chunk_id, {})

        # Build the slot dict expected by build_user_prompt_conditioned
        s3_meta   = chunk_rec.get("metadata", {})
        fringe    = chunk_rec.get("context_fringe", {})
        slot = {
            "text":          chunk_rec.get("text", rec.get("text", "")),
            "bloom_level":   rec.get("bloom_level", s3_meta.get("bloom_level", "understand")),
            "chunk_type":    rec.get("chunk_type",  s3_meta.get("chunk_type", "definition")),
            "concept":       rec.get("concept",     (s3_meta.get("concepts") or [""])[0]),
            "keywords":      s3_meta.get("keywords", []),
            "prev_sentence": (fringe.get("prev_sentence") or "").strip(),
            "next_sentence": (fringe.get("next_sentence") or "").strip(),
        }

        messages = [
            {"role": "system", "content": SFT_SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt_conditioned(slot)},
        ]

        generated_mcq = _run_student_inference(model, tokenizer, messages)

        output_rec = {
            "chunk_id":   chunk_id,
            "slot_index": slot_index,
            "text":       slot["text"],
            "chunk_type": slot["chunk_type"],
            "concept":    slot["concept"],
            "bloom_level":slot["bloom_level"],
            "stage":      stage_label,
            "mcq": generated_mcq if isinstance(generated_mcq, dict) and "_raw" not in generated_mcq
                   else {"question": "", "options": {}, "answer": "", "explanation": "",
                         "_raw_unparsed": generated_mcq.get("_raw", "")},
        }
        append_jsonl(output_path, output_rec)
        all_outputs.append(output_rec)

    print(f"\n✓ Generation complete: {len(all_outputs):,} total records in {output_path}")
    return all_outputs


# ── Run pre-FT inference ──────────────────────────────────────────────────────
pre_ft_outputs = generate_student_outputs(
    model          = pre_model,
    tokenizer      = pre_tokenizer,
    teacher_eval_records = teacher_eval_records,
    chunk_lookup   = chunk_lookup,
    output_path    = STUDENT_PRE_OUTPUT_PATH,
    stage_label    = "student_pre_ft",
)

# ── Unload pre-FT model to free GPU memory ────────────────────────────────────
del pre_model
gc.collect()
torch.cuda.empty_cache()
print("✓ Pre-FT model unloaded from GPU.")

In [160]:
def _run_student_inference(
    model, tokenizer, messages: List[Dict], max_new_tokens: int = 512
) -> dict:
    """Run a single inference call. Returns parsed MCQ dict or raw str."""
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    new_ids = out_ids[0][inputs.input_ids.shape[1]:]
    raw     = tokenizer.decode(new_ids, skip_special_tokens=True)
    try:
        clean = raw.strip()
        clean = re.sub(r"^```json\s*", "", clean)
        clean = re.sub(r"^```\s*", "", clean)
        clean = re.sub(r"\s*```$", "", clean)
        return json.loads(clean)
    except json.JSONDecodeError:
        return {"_raw": raw}


def generate_student_outputs(
    model, tokenizer,
    teacher_eval_records: List[Dict],
    chunk_lookup:         Dict[str, Dict],
    output_path:          str,
    stage_label:          str,
) -> List[Dict]:
    """
    Generate MCQ outputs for all teacher eval records using the given model.
    Uses the EXACT same SFT prompt format (SFT_SYSTEM_PROMPT + build_user_prompt_conditioned).
    Fully resumable: skips already-generated samples.
    Returns list of all output records.
    """
    # Load already-done IDs
    existing     = load_jsonl_safe(output_path)
    done_keys    = {record_key(r) for r in existing}
    all_outputs  = existing[:]
    to_generate  = [r for r in teacher_eval_records if record_key(r) not in done_keys]

    print(f"\n{'='*60}")
    print(f"Stage         : {stage_label}")
    print(f"Total records : {len(teacher_eval_records):,}")
    print(f"Already done  : {len(done_keys):,}")
    print(f"To generate   : {len(to_generate):,}")
    print(f"Output file   : {output_path}")
    print(f"{'='*60}")

    if not to_generate:
        print("✓ All outputs already generated.")
        return all_outputs

    for rec in tqdm(to_generate, desc=f"Generating [{stage_label}]"):
        chunk_id   = rec.get("chunk_id", "")
        slot_index = rec.get("slot_index", 0)
        chunk_rec  = chunk_lookup.get(chunk_id, {})

        # Build the slot dict expected by build_user_prompt_conditioned
        s3_meta   = chunk_rec.get("metadata", {})
        fringe    = chunk_rec.get("context_fringe", {})
        slot = {
            "text":          chunk_rec.get("text", rec.get("text", "")),
            "bloom_level":   rec.get("bloom_level", s3_meta.get("bloom_level", "understand")),
            "chunk_type":    rec.get("chunk_type",  s3_meta.get("chunk_type", "definition")),
            "concept":       rec.get("concept",     (s3_meta.get("concepts") or [""])[0]),
            "keywords":      s3_meta.get("keywords", []),
            "prev_sentence": (fringe.get("prev_sentence") or "").strip(),
            "next_sentence": (fringe.get("next_sentence") or "").strip(),
        }

        messages = [
            {"role": "system", "content": SFT_SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt_conditioned(slot)},
        ]

        generated_mcq = _run_student_inference(model, tokenizer, messages)

        output_rec = {
            "chunk_id":   chunk_id,
            "slot_index": slot_index,
            "text":       slot["text"],
            "chunk_type": slot["chunk_type"],
            "concept":    slot["concept"],
            "bloom_level":slot["bloom_level"],
            "stage":      stage_label,
            "mcq": generated_mcq if isinstance(generated_mcq, dict) and "_raw" not in generated_mcq
                   else {"question": "", "options": {}, "answer": "", "explanation": "",
                         "_raw_unparsed": generated_mcq.get("_raw", "")},
        }
        append_jsonl(output_path, output_rec)
        all_outputs.append(output_rec)

    print(f"\n✓ Generation complete: {len(all_outputs):,} total records in {output_path}")
    return all_outputs

### 12.7  Student Model Inference — Post-Fine-Tuning

Generates MCQs using the **merged fine-tuned Qwen model** (after LoRA fine-tuning).
Uses the same prompt format. Fully resumable.


In [161]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("HF token loaded!")

HF token loaded!


In [162]:
from huggingface_hub import model_info
model_info("Aya-khaled/mcq-bloom-qwen-merged_v2", token=HF_TOKEN)

ModelInfo(id='Aya-khaled/mcq-bloom-qwen-merged_v2', author='Aya-khaled', sha='11e538e6f7f0b6b5f24f7176254327f717d856dd', created_at=datetime.datetime(2026, 5, 23, 16, 1, 33, tzinfo=datetime.timezone.utc), last_modified=datetime.datetime(2026, 5, 23, 23, 43, 11, tzinfo=datetime.timezone.utc), private=True, disabled=False, downloads=0, downloads_all_time=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, likes=0, library_name='transformers', tags=['transformers', 'safetensors', 'qwen2', 'text-generation', 'conversational', 'arxiv:1910.09700', 'text-generation-inference', 'endpoints_compatible', 'region:us'], pipeline_tag='text-generation', mask_token=None, card_data={'base_model': None, 'datasets': None, 'eval_results': None, 'language': None, 'library_name': 'transformers', 'license': None, 'license_name': None, 'license_link': None, 'metrics': None, 'model_name': None, 'pipeline_tag': None, 'tags': []}, widget_data=[{'text': 'Hi, what can you help me with?'}

In [129]:
import os
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

KAGGLE_BEST_MERGED = "/kaggle/working/mcq-qwen-merged-best_v2"

HF_MERGED_ID = "Aya-khaled/mcq-bloom-qwen-merged_v2"

# Load local merged model only if it exists
if os.path.exists(KAGGLE_BEST_MERGED):
    post_ft_model_path = KAGGLE_BEST_MERGED
    print(f"Loading post-FT model from local merged model: {post_ft_model_path}")

# Otherwise load merged HF model
else:
    post_ft_model_path = HF_MERGED_ID
    print(f"Loading post-FT model from HuggingFace: {post_ft_model_path}")

post_tokenizer = AutoTokenizer.from_pretrained(
    post_ft_model_path,
    token=HF_TOKEN
)

post_model = AutoModelForCausalLM.from_pretrained(
    post_ft_model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_TOKEN,
)

post_model.eval()

print("✓ Post-FT model loaded")

Loading post-FT model from local merged model: /kaggle/working/mcq-qwen-merged-best_v2


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✓ Post-FT model loaded


In [130]:
# ── Run post-FT inference ─────────────────────────────────────────────────────
post_ft_outputs = generate_student_outputs(
    model          = post_model,
    tokenizer      = post_tokenizer,
    teacher_eval_records = teacher_eval_records,
    chunk_lookup   = chunk_lookup,
    output_path    = STUDENT_POST_OUTPUT_PATH,
    stage_label    = "student_post_ft",
)

# ── Unload post-FT model ──────────────────────────────────────────────────────
del post_model
gc.collect()
torch.cuda.empty_cache()
print("✓ Post-FT model unloaded from GPU.")


Stage         : student_post_ft
Total records : 100
Already done  : 0
To generate   : 100
Output file   : /kaggle/working/evaluation/student_post_ft_outputs.jsonl


Generating [student_post_ft]:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



✓ Generation complete: 100 total records in /kaggle/working/evaluation/student_post_ft_outputs.jsonl
✓ Post-FT model unloaded from GPU.


In [50]:
# # ─── 12.7  Student Inference — Post-Fine-Tuning ──────────────────────────────
# import torch, gc
# from transformers import AutoModelForCausalLM, AutoTokenizer

# # ── Resolve post-FT model path ────────────────────────────────────────────────
# KAGGLE_BEST_MERGED = "/kaggle/working/mcq-qwen-merged-best"
# KAGGLE_MERGED_DIR  = "/kaggle/working/mcq-qwen-merged"
# # HF_MERGED_ID       = "Quiz-Generation/mcq-bloom-qwen-merged"  # <-- update to your HF repo
# HF_MERGED_ID       = "https://huggingface.co/Quiz-Generation/mcq-bloom-qwen-merged"

# if os.path.exists(KAGGLE_BEST_MERGED):
#     post_ft_model_path = KAGGLE_BEST_MERGED
#     print(f"Loading post-FT model from local best: {post_ft_model_path}")
# elif os.path.exists(KAGGLE_MERGED_DIR):
#     post_ft_model_path = KAGGLE_MERGED_DIR
#     print(f"Loading post-FT model from local merged: {post_ft_model_path}")
# else:
#     post_ft_model_path = HF_MERGED_ID
#     print(f"Loading post-FT model from HuggingFace: {post_ft_model_path}")

# post_tokenizer = AutoTokenizer.from_pretrained(post_ft_model_path, token=HF_TOKEN)
# post_model     = AutoModelForCausalLM.from_pretrained(
#     post_ft_model_path,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     token=HF_TOKEN,
# )
# post_model.eval()
# print(f"✓ Post-FT model loaded on {DEVICE}")

# # ── Run post-FT inference ─────────────────────────────────────────────────────
# post_ft_outputs = generate_student_outputs(
#     model          = post_model,
#     tokenizer      = post_tokenizer,
#     teacher_eval_records = teacher_eval_records,
#     chunk_lookup   = chunk_lookup,
#     output_path    = STUDENT_POST_OUTPUT_PATH,
#     stage_label    = "student_post_ft",
# )

# # ── Unload post-FT model ──────────────────────────────────────────────────────
# del post_model
# gc.collect()
# torch.cuda.empty_cache()
# print("✓ Post-FT model unloaded from GPU.")

### 12.8  Run LLM Judge on All Three Stages

Evaluates each model's outputs independently. Each stage is fully resumable.


In [26]:
# ─── 12.8  Run LLM Judge — All Three Stages ──────────────────────────────────

# Install openai if not available
try:
    import openai
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    import openai

print(f"Using judge model: {JUDGE_MODEL}")
print()

# ── Stage 1: Teacher ──────────────────────────────────────────────────────────
teacher_scores = evaluate_stage(
    records      = teacher_eval_records,
    chunk_lookup = chunk_lookup,
    scores_path  = TEACHER_SCORES_PATH,
    stage_label  = "teacher",
)

Using judge model: meta-llama/llama-3.3-70b-instruct


Stage        : teacher
Total records: 100
Already done : 0
To evaluate  : 100
Scores file  : /kaggle/working/evaluation/teacher_judge_scores.jsonl


Judging [teacher]:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Done.  Skipped=0  Failed=9  Scored=91


In [35]:
# ── Stage 2: Student Pre-FT ───────────────────────────────────────────────────
# Filter pre_ft_outputs to only records that have a valid MCQ (skip parse failures)
valid_pre_ft = [
    r for r in pre_ft_outputs
    if r.get("mcq", {}).get("question")
]
print(f"\nPre-FT valid MCQs for judging: {len(valid_pre_ft):,} / {len(pre_ft_outputs):,}")

    records      = valid_pre_ft,
    chunk_lookup = chunk_lookup,
    scores_path  = STUDENT_PRE_SCORES_PATH,
    stage_label  = "student_pre_ft",
)


Pre-FT valid MCQs for judging: 95 / 100

Stage        : student_pre_ft
Total records: 95
Already done : 0
To evaluate  : 95
Scores file  : /kaggle/working/evaluation/student_pre_ft_judge_scores.jsonl


Judging [student_pre_ft]:   0%|          | 0/95 [00:00<?, ?it/s]


✓ Done.  Skipped=0  Failed=12  Scored=83


In [184]:
import json

file_path = "/kaggle/working/evaluation/student_post_ft_judge_scores.jsonl"

cleaned_lines = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        if obj.get("status") == "judge_parse_failed":
            continue

        cleaned_lines.append(obj)

with open(file_path, "w", encoding="utf-8") as f:
    for obj in cleaned_lines:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"✓ Done cleaning file")
print(f"Remaining records: {len(cleaned_lines)}")

✓ Done cleaning file
Remaining records: 63


In [185]:
# ── Stage 3: Student Post-FT ──────────────────────────────────────────────────
valid_post_ft = [
    r for r in post_ft_outputs
    if r.get("mcq", {}).get("question")
]
print(f"\nPost-FT valid MCQs for judging: {len(valid_post_ft):,} / {len(post_ft_outputs):,}")

student_post_scores = evaluate_stage(
    records      = valid_post_ft,
    chunk_lookup = chunk_lookup,
    scores_path  = STUDENT_POST_SCORES_PATH,
    stage_label  = "student_post_ft",
)


Post-FT valid MCQs for judging: 99 / 100

Stage        : student_post_ft
Total records: 99
Already done : 63
To evaluate  : 36
Scores file  : /kaggle/working/evaluation/student_post_ft_judge_scores.jsonl


Judging [student_post_ft]:   0%|          | 0/36 [00:00<?, ?it/s]


✓ Done.  Skipped=0  Failed=10  Scored=26


In [53]:
print("\n" + "="*60)
print("✓ All three stages evaluated.")
print(f"  Teacher scores    : {len(teacher_scores):,} records → {TEACHER_SCORES_PATH}")
print(f"  Pre-FT scores     : {len(student_pre_scores):,} records → {STUDENT_PRE_SCORES_PATH}")
print(f"  Post-FT scores    : {len(student_post_scores):,} records → {STUDENT_POST_SCORES_PATH}")


✓ All three stages evaluated.
  Teacher scores    : 100 records → /kaggle/working/evaluation/teacher_judge_scores.jsonl
  Pre-FT scores     : 95 records → /kaggle/working/evaluation/student_pre_ft_judge_scores.jsonl
  Post-FT scores    : 98 records → /kaggle/working/evaluation/student_post_ft_judge_scores.jsonl


In [ ]:
# # ─── 12.8  Run LLM Judge — All Three Stages ──────────────────────────────────

# # Install openai if not available
# try:
#     import openai
# except ImportError:
#     import subprocess, sys
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
#     import openai

# print(f"Using judge model: {JUDGE_MODEL}")
# print()

# # ── Stage 1: Teacher ──────────────────────────────────────────────────────────
# teacher_scores = evaluate_stage(
#     records      = teacher_eval_records,
#     chunk_lookup = chunk_lookup,
#     scores_path  = TEACHER_SCORES_PATH,
#     stage_label  = "teacher",
# )

# # ── Stage 2: Student Pre-FT ───────────────────────────────────────────────────
# # Filter pre_ft_outputs to only records that have a valid MCQ (skip parse failures)
# valid_pre_ft = [
#     r for r in pre_ft_outputs
#     if r.get("mcq", {}).get("question")
# ]
# print(f"\nPre-FT valid MCQs for judging: {len(valid_pre_ft):,} / {len(pre_ft_outputs):,}")

# student_pre_scores = evaluate_stage(
#     records      = valid_pre_ft,
#     chunk_lookup = chunk_lookup,
#     scores_path  = STUDENT_PRE_SCORES_PATH,
#     stage_label  = "student_pre_ft",
# )

# # ── Stage 3: Student Post-FT ──────────────────────────────────────────────────
# valid_post_ft = [
#     r for r in post_ft_outputs
#     if r.get("mcq", {}).get("question")
# ]
# print(f"\nPost-FT valid MCQs for judging: {len(valid_post_ft):,} / {len(post_ft_outputs):,}")

# student_post_scores = evaluate_stage(
#     records      = valid_post_ft,
#     chunk_lookup = chunk_lookup,
#     scores_path  = STUDENT_POST_SCORES_PATH,
#     stage_label  = "student_post_ft",
# )

# print("\n" + "="*60)
# print("✓ All three stages evaluated.")
# print(f"  Teacher scores    : {len(teacher_scores):,} records → {TEACHER_SCORES_PATH}")
# print(f"  Pre-FT scores     : {len(student_pre_scores):,} records → {STUDENT_PRE_SCORES_PATH}")
# print(f"  Post-FT scores    : {len(student_post_scores):,} records → {STUDENT_POST_SCORES_PATH}")


In [189]:
# ─── Load Existing Evaluation Score Files ────────────────────────────────────
import os
import json
from typing import List, Dict

# ── Paths ────────────────────────────────────────────────────────────────────
SOURCE_DIR = "/kaggle/input/datasets/ayakhaled51/mcqs-files/"

TEACHER_SCORES_PATH = os.path.join(SOURCE_DIR, "teacher_judge_scores.jsonl")
STUDENT_PRE_SCORES_PATH = os.path.join(SOURCE_DIR, "student_pre_ft_judge_scores.jsonl")

# ── Output dirs (used later by your comparison/plot cells) ──────────────────
KAGGLE_WORKING = "/kaggle/working"

EVAL_DIR = os.path.join(KAGGLE_WORKING, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)

COMPARISON_PATH = os.path.join(EVAL_DIR, "final_comparison_results.json")

PLOT_DIR = os.path.join(EVAL_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Helper ───────────────────────────────────────────────────────────────────
def load_jsonl_safe(path: str) -> List[Dict]:
    records = []

    if not os.path.exists(path):
        print(f"File not found: {path}")
        return records

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass

    return records

# ── Load the 3 evaluation score files ───────────────────────────────────────
teacher_scores = load_jsonl_safe(TEACHER_SCORES_PATH)
student_pre_scores = load_jsonl_safe(STUDENT_PRE_SCORES_PATH)

print("✓ Evaluation score files loaded:")
print(f"  Teacher scores     : {len(teacher_scores):,}")
print(f"  Student Pre-FT     : {len(student_pre_scores):,}")

✓ Evaluation score files loaded:
  Teacher scores     : 100
  Student Pre-FT     : 95


### 12.9  Compute Stage Statistics & Quality Preservation

In [190]:
# ─── 12.9  Compute Stage Statistics & Quality Preservation ───────────────────
import statistics, json

DIM_KEYS = [
    "D1_factual_grounding",
    "D2_question_clarity",
    "D3_correct_answer_validity",
    "D4_distractor_plausibility",
    "D5_bloom_alignment",
    "D6_topic_lo_alignment",
]

DIM_LABELS = {
    "D1_factual_grounding":       "D1 — Factual Grounding",
    "D2_question_clarity":        "D2 — Question Clarity",
    "D3_correct_answer_validity": "D3 — Correct Answer Validity",
    "D4_distractor_plausibility": "D4 — Distractor Plausibility",
    "D5_bloom_alignment":         "D5 — Bloom Alignment",
    "D6_topic_lo_alignment":      "D6 — Topic & LO Alignment",
}

def compute_stage_stats(score_records: List[Dict]) -> Dict:
    """Compute per-dimension and CMQS mean/std/median for valid (status=ok) records."""
    valid = [r for r in score_records if r.get("status") == "ok"]
    if not valid:
        return {}

    stats = {}

    # CMQS overall
    cmqs_vals = [r["CMQS"] for r in valid if r.get("CMQS") is not None]
    stats["CMQS"] = {
        "mean":   round(statistics.mean(cmqs_vals), 4),
        "std":    round(statistics.stdev(cmqs_vals) if len(cmqs_vals) > 1 else 0.0, 4),
        "median": round(statistics.median(cmqs_vals), 4),
        "min":    round(min(cmqs_vals), 4),
        "max":    round(max(cmqs_vals), 4),
        "n":      len(cmqs_vals),
    }

    # Per dimension
    for key in DIM_KEYS:
        vals = []
        for r in valid:
            dim = (r.get("scores") or {}).get(key, {})
            s   = dim.get("score") if isinstance(dim, dict) else dim
            if s is not None:
                vals.append(float(s))
        if vals:
            stats[key] = {
                "mean":   round(statistics.mean(vals), 4),
                "std":    round(statistics.stdev(vals) if len(vals) > 1 else 0.0, 4),
                "median": round(statistics.median(vals), 4),
                "n":      len(vals),
            }

    # Bloom-level CMQS breakdown
    bloom_cmqs = {}
    for r in valid:
        bl  = r.get("bloom_level", "unknown")
        val = r.get("CMQS")
        if val is not None:
            bloom_cmqs.setdefault(bl, []).append(val)
    stats["bloom_breakdown"] = {
        bl: {
            "mean": round(statistics.mean(vs), 4),
            "std":  round(statistics.stdev(vs) if len(vs) > 1 else 0.0, 4),
            "n":    len(vs),
        }
        for bl, vs in sorted(bloom_cmqs.items())
    }

    # Quality label distribution
    label_counts = {"Excellent": 0, "Good": 0, "Acceptable": 0, "Poor": 0, "Reject": 0}
    for v in cmqs_vals:
        if   v >= 4.50: label_counts["Excellent"]  += 1
        elif v >= 3.75: label_counts["Good"]        += 1
        elif v >= 3.00: label_counts["Acceptable"]  += 1
        elif v >= 2.00: label_counts["Poor"]        += 1
        else:           label_counts["Reject"]      += 1
    stats["quality_distribution"] = {
        k: {"count": v, "pct": round(v / len(cmqs_vals) * 100, 1)}
        for k, v in label_counts.items()
    }

    stats["n_valid"] = len(valid)
    stats["n_total"] = len(score_records)
    return stats


# ── Compute stats for all three stages ───────────────────────────────────────
stats_teacher  = compute_stage_stats(teacher_scores)
stats_pre_ft   = compute_stage_stats(student_pre_scores)
stats_post_ft  = compute_stage_stats(student_post_scores)

# ── Quality Preservation Score ────────────────────────────────────────────────
def qps(post_val, teacher_val):
    if teacher_val and teacher_val > 0:
        return round(post_val / teacher_val * 100, 2)
    return None

teacher_cmqs  = stats_teacher.get("CMQS", {}).get("mean", 0)
post_ft_cmqs  = stats_post_ft.get("CMQS", {}).get("mean", 0)
pre_ft_cmqs   = stats_pre_ft.get("CMQS", {}).get("mean", 0)

overall_qps = qps(post_ft_cmqs, teacher_cmqs)
ft_gain     = round(post_ft_cmqs - pre_ft_cmqs, 4) if pre_ft_cmqs else None

print("\n" + "="*60)
print("EVALUATION RESULTS — SUMMARY")
print("="*60)
print(f"{'Metric':<35} {'Teacher':>9} {'Pre-FT':>9} {'Post-FT':>9}")
print("-"*62)
print(f"{'CMQS (mean ± std)':<35} "
      f"{stats_teacher.get('CMQS',{}).get('mean','-'):>6.3f} ± {stats_teacher.get('CMQS',{}).get('std',0):>4.3f}  "
      f"{stats_pre_ft.get('CMQS',{}).get('mean','-'):>6.3f} ± {stats_pre_ft.get('CMQS',{}).get('std',0):>4.3f}  "
      f"{stats_post_ft.get('CMQS',{}).get('mean','-'):>6.3f} ± {stats_post_ft.get('CMQS',{}).get('std',0):>4.3f}")
print()
for key in DIM_KEYS:
    t = stats_teacher.get(key, {}).get("mean", "-")
    b = stats_pre_ft.get(key, {}).get("mean", "-")
    a = stats_post_ft.get(key, {}).get("mean", "-")
    label = DIM_LABELS[key]
    print(f"  {label:<33} {t:>9.3f}  {b:>9.3f}  {a:>9.3f}")
print()
print(f"Quality Preservation Score (QPS):   {overall_qps}%")
print(f"Fine-Tuning Gain (Post - Pre CMQS): {ft_gain:+.4f}" if ft_gain is not None else "N/A")
print("="*60)

# ── Save all stats ────────────────────────────────────────────────────────────
all_stats = {
    "teacher":         stats_teacher,
    "student_pre_ft":  stats_pre_ft,
    "student_post_ft": stats_post_ft,
    "summary": {
        "QPS_overall_pct":      overall_qps,
        "FT_gain_CMQS":         ft_gain,
        "teacher_CMQS_mean":    teacher_cmqs,
        "pre_ft_CMQS_mean":     pre_ft_cmqs,
        "post_ft_CMQS_mean":    post_ft_cmqs,
        "judge_model":          JUDGE_MODEL,
        "eval_sample_n":        EVAL_SAMPLE_N,
    }
}
with open(COMPARISON_PATH, "w", encoding="utf-8") as f:
    json.dump(all_stats, f, indent=2, ensure_ascii=False)
print(f"\n✓ Full comparison results saved → {COMPARISON_PATH}")



EVALUATION RESULTS — SUMMARY
Metric                                Teacher    Pre-FT   Post-FT
--------------------------------------------------------------
CMQS (mean ± std)                    4.706 ± 0.151   4.066 ± 0.887   4.354 ± 0.546

  D1 — Factual Grounding                4.758      3.788      4.045
  D2 — Question Clarity                 5.000      4.753      4.955
  D3 — Correct Answer Validity          5.000      3.929      4.472
  D4 — Distractor Plausibility          3.978      3.682      3.832
  D5 — Bloom Alignment                  4.813      4.247      4.596
  D6 — Topic & LO Alignment             4.989      4.847      4.978

Quality Preservation Score (QPS):   92.53%
Fine-Tuning Gain (Post - Pre CMQS): +0.2880

✓ Full comparison results saved → /kaggle/working/evaluation/final_comparison_results.json


### 12.10  Visualizations

In [191]:
# ─── 12.10  Visualizations ───────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

PLOT_DIR = os.path.join(EVAL_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

STAGE_COLORS  = {"teacher": "#2563EB", "student_pre_ft": "#DC2626", "student_post_ft": "#16A34A"}
STAGE_LABELS  = {"teacher": "Teacher (Gemini)", "student_pre_ft": "Student Pre-FT (Base Qwen)",
                 "student_post_ft": "Student Post-FT (Distilled Qwen)"}
SHORT_DIM     = ["D1\nGrounding", "D2\nClarity", "D3\nAnswer\nValidity",
                 "D4\nDistractor\nPlausibility", "D5\nBloom\nAlignment", "D6\nTopic\nAlignment"]

# ── V1: Radar Chart ───────────────────────────────────────────────────────────
def make_radar(stats_dict_map, title, save_path):
    dims   = DIM_KEYS
    labels = [d.split("_", 1)[1].replace("_", " ").title() for d in dims]
    N      = len(dims)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
    for stage, stats in stats_dict_map.items():
        values = [stats.get(d, {}).get("mean", 0) for d in dims]
        values += values[:1]
        ax.plot(angles, values, "o-", linewidth=2,
                color=STAGE_COLORS[stage], label=STAGE_LABELS[stage])
        ax.fill(angles, values, alpha=0.08, color=STAGE_COLORS[stage])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=11)
    ax.set_ylim(0, 5)
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_title(title, size=14, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {save_path}")

make_radar(
    {"teacher": stats_teacher, "student_pre_ft": stats_pre_ft, "student_post_ft": stats_post_ft},
    "MCQ Quality Dimensions — All Three Stages",
    os.path.join(PLOT_DIR, "V1_radar_all_stages.png")
)

# ── V2: Grouped Bar Chart ─────────────────────────────────────────────────────
def make_grouped_bar(stats_dict_map, title, save_path):
    dim_means  = {stage: [s.get(d, {}).get("mean", 0) for d in DIM_KEYS]
                  for stage, s in stats_dict_map.items()}
    dim_stds   = {stage: [s.get(d, {}).get("std", 0) for d in DIM_KEYS]
                  for stage, s in stats_dict_map.items()}

    x      = np.arange(len(DIM_KEYS))
    width  = 0.25
    stages = list(stats_dict_map.keys())
    offsets= [-width, 0, width]

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, stage in enumerate(stages):
        ax.bar(x + offsets[i], dim_means[stage], width,
               yerr=dim_stds[stage], capsize=4,
               color=STAGE_COLORS[stage], label=STAGE_LABELS[stage],
               alpha=0.85, error_kw={"elinewidth": 1.5})

    ax.set_xticks(x)
    ax.set_xticklabels(SHORT_DIM, fontsize=10)
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("Mean Score (1–5)", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.axhline(3.0, color="gray", ls="--", lw=1, alpha=0.5, label="Acceptable threshold")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")

make_grouped_bar(
    {"teacher": stats_teacher, "student_pre_ft": stats_pre_ft, "student_post_ft": stats_post_ft},
    "Per-Dimension MCQ Quality — Mean ± Std (All Stages)",
    os.path.join(PLOT_DIR, "V2_grouped_bar_dimensions.png")
)

# ── V3: Bloom-Level Line Chart ────────────────────────────────────────────────
def make_bloom_line(stats_dict_map, title, save_path):
    bloom_levels = ["remember", "understand", "apply", "analyze", "evaluate", "create"]
    fig, ax = plt.subplots(figsize=(10, 5))
    for stage, stats in stats_dict_map.items():
        bb = stats.get("bloom_breakdown", {})
        means = [bb.get(bl, {}).get("mean", np.nan) for bl in bloom_levels]
        ax.plot(bloom_levels, means, "o-", linewidth=2.5,
                color=STAGE_COLORS[stage], label=STAGE_LABELS[stage], markersize=7)
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("CMQS Mean", fontsize=12)
    ax.set_xlabel("Bloom Taxonomy Level", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.axhline(3.0, color="gray", ls="--", lw=1, alpha=0.5)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")

make_bloom_line(
    {"teacher": stats_teacher, "student_pre_ft": stats_pre_ft, "student_post_ft": stats_post_ft},
    "CMQS by Bloom Taxonomy Level — All Stages",
    os.path.join(PLOT_DIR, "V3_bloom_level_lines.png")
)

# ── V4: Quality Distribution Stacked Bar ──────────────────────────────────────
def make_quality_dist(stats_dict_map, title, save_path):
    labels_order = ["Excellent", "Good", "Acceptable", "Poor", "Reject"]
    colors_q     = ["#15803d", "#65a30d", "#ca8a04", "#dc2626", "#7f1d1d"]
    stages       = list(stats_dict_map.keys())
    stage_labels = [STAGE_LABELS[s] for s in stages]

    data = {lbl: [] for lbl in labels_order}
    for stage in stages:
        qd = stats_dict_map[stage].get("quality_distribution", {})
        for lbl in labels_order:
            data[lbl].append(qd.get(lbl, {}).get("pct", 0))

    fig, ax = plt.subplots(figsize=(9, 5))
    bottoms = [0.0] * len(stages)
    for lbl, color in zip(labels_order, colors_q):
        vals = data[lbl]
        ax.bar(stage_labels, vals, bottom=bottoms, label=lbl, color=color, alpha=0.85, width=0.5)
        for i, (v, b) in enumerate(zip(vals, bottoms)):
            if v > 4:
                ax.text(i, b + v / 2, f"{v:.0f}%", ha="center", va="center",
                        fontsize=10, color="white", fontweight="bold")
        bottoms = [b + v for b, v in zip(bottoms, vals)]

    ax.set_ylim(0, 105)
    ax.set_ylabel("Percentage of MCQs", fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")

make_quality_dist(
    {"teacher": stats_teacher, "student_pre_ft": stats_pre_ft, "student_post_ft": stats_post_ft},
    "MCQ Quality Label Distribution per Stage",
    os.path.join(PLOT_DIR, "V4_quality_distribution.png")
)

# ── V5: Gap Heatmap ───────────────────────────────────────────────────────────
def make_gap_heatmap(stats_teacher, stats_post_ft, stats_pre_ft, title, save_path):
    rows    = [DIM_LABELS[d] for d in DIM_KEYS] + ["CMQS (Composite)"]
    all_keys= DIM_KEYS + ["CMQS"]

    t_means  = [stats_teacher.get(k, {}).get("mean", 0) for k in all_keys]
    b_means  = [stats_pre_ft.get(k, {}).get("mean", 0)  for k in all_keys]
    a_means  = [stats_post_ft.get(k, {}).get("mean", 0) for k in all_keys]

    gap_ta   = [round(t - a, 3) for t, a in zip(t_means, a_means)]  # Teacher - Post-FT
    gain_ba  = [round(a - b, 3) for b, a in zip(b_means, a_means)]  # Post-FT - Pre-FT

    data = np.array([gap_ta, gain_ba])  # shape (2, 7)
    fig, ax = plt.subplots(figsize=(12, 4))
    im  = ax.imshow(data, cmap="RdYlGn_r", aspect="auto", vmin=-1.5, vmax=1.5)
    plt.colorbar(im, ax=ax, label="Score difference")

    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(rows, rotation=30, ha="right", fontsize=10)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["Teacher − Post-FT gap\n(red=large gap)", "Post-FT − Pre-FT gain\n(green=improvement)"],
                       fontsize=10)
    for j, (g1, g2) in enumerate(zip(gap_ta, gain_ba)):
        ax.text(j, 0, f"{g1:+.2f}", ha="center", va="center", fontsize=9, color="black")
        ax.text(j, 1, f"{g2:+.2f}", ha="center", va="center", fontsize=9, color="black")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")

make_gap_heatmap(
    stats_teacher, stats_post_ft, stats_pre_ft,
    "Gap Heatmap: Teacher–Student Quality Differences",
    os.path.join(PLOT_DIR, "V5_gap_heatmap.png")
)

# ── V6: CMQS Comparison Bar ───────────────────────────────────────────────────
def make_cmqs_bar(stats_dict_map, qps_val, ft_gain_val, save_path):
    stages  = list(stats_dict_map.keys())
    means   = [stats_dict_map[s].get("CMQS", {}).get("mean", 0) for s in stages]
    stds    = [stats_dict_map[s].get("CMQS", {}).get("std",  0) for s in stages]
    colors  = [STAGE_COLORS[s] for s in stages]
    xlabels = [STAGE_LABELS[s] for s in stages]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(xlabels, means, yerr=stds, capsize=6, color=colors, alpha=0.85,
                  width=0.5, error_kw={"elinewidth": 2})
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, mean + 0.05, f"{mean:.3f}",
                ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("CMQS Mean ± Std", fontsize=12)
    ax.set_title(
        f"Composite MCQ Quality Score (CMQS) — All Stages\n"
        f"Quality Preservation: QPS={qps_val}%  |  FT Gain={ft_gain_val:+.3f}",
        fontsize=12
    )
    ax.axhline(3.0, color="gray", ls="--", lw=1.5, alpha=0.6, label="Acceptable (3.0)")
    ax.axhline(3.75, color="goldenrod", ls="--", lw=1.5, alpha=0.6, label="Good (3.75)")
    ax.axhline(4.50, color="green", ls="--", lw=1.5, alpha=0.6, label="Excellent (4.5)")
    ax.legend(fontsize=9, loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"  Saved: {save_path}")

make_cmqs_bar(
    {"teacher": stats_teacher, "student_pre_ft": stats_pre_ft, "student_post_ft": stats_post_ft},
    qps_val=overall_qps,
    ft_gain_val=ft_gain or 0,
    save_path=os.path.join(PLOT_DIR, "V6_cmqs_bar.png")
)

print(f"\n✓ All plots saved to: {PLOT_DIR}")


  Saved: /kaggle/working/evaluation/plots/V1_radar_all_stages.png
  Saved: /kaggle/working/evaluation/plots/V2_grouped_bar_dimensions.png
  Saved: /kaggle/working/evaluation/plots/V3_bloom_level_lines.png
  Saved: /kaggle/working/evaluation/plots/V4_quality_distribution.png
  Saved: /kaggle/working/evaluation/plots/V5_gap_heatmap.png
  Saved: /kaggle/working/evaluation/plots/V6_cmqs_bar.png

✓ All plots saved to: /kaggle/working/evaluation/plots


### 12.11  Final Comparison Report

In [192]:
# ─── 12.11  Final Comparison Report ─────────────────────────────────────────

print("\n" + "╔" + "═"*64 + "╗")
print("║" + "  AGENTIC AI TUTOR — FINAL EVALUATION COMPARISON REPORT".center(64) + "║")
print("╚" + "═"*64 + "╝")

print(f"\n  Judge model    : {JUDGE_MODEL}")
print(f"  Eval sample N  : {EVAL_SAMPLE_N if EVAL_SAMPLE_N else 'ALL'}")
print(f"  Teacher N      : {stats_teacher.get('n_valid', 0):,} valid MCQs scored")
print(f"  Pre-FT N       : {stats_pre_ft.get('n_valid', 0):,} valid MCQs scored")
print(f"  Post-FT N      : {stats_post_ft.get('n_valid', 0):,} valid MCQs scored")

print("\n── Main Results ─────────────────────────────────────────────────────────")
header = f"  {'Dimension':<35} {'Teacher':>9} {'Pre-FT':>9} {'Post-FT':>9}  {'FT Gain':>8}  {'QPS %':>7}"
print(header)
print("  " + "─"*74)

all_keys_display = DIM_KEYS + ["CMQS"]
all_labels_display = {**DIM_LABELS, "CMQS": "CMQS (Composite, weighted)"}

for key in all_keys_display:
    t = stats_teacher.get(key, {}).get("mean", 0)
    b = stats_pre_ft.get(key, {}).get("mean", 0)
    a = stats_post_ft.get(key, {}).get("mean", 0)
    gain = a - b
    q    = qps(a, t)
    lbl  = all_labels_display.get(key, key)
    marker = " ←★" if key == "CMQS" else ""
    print(f"  {lbl:<35} {t:>9.3f}  {b:>9.3f}  {a:>9.3f}  {gain:>+8.3f}  {q:>6.1f}%{marker}")

print("\n── Quality Preservation ─────────────────────────────────────────────────")
print(f"  Quality Preservation Score (QPS) = Post-FT / Teacher × 100")
print(f"  QPS = {post_ft_cmqs:.3f} / {teacher_cmqs:.3f} × 100 = {overall_qps:.1f}%")

if overall_qps >= 90:
    qps_interpretation = "EXCELLENT — Student matches teacher quality closely."
elif overall_qps >= 80:
    qps_interpretation = "GOOD — Fine-tuning is effective; minor gap from teacher."
elif overall_qps >= 70:
    qps_interpretation = "MODERATE — Acceptable; specific dimensions may need improvement."
elif overall_qps >= 60:
    qps_interpretation = "WEAK — Fine-tuning insufficient; review distillation pipeline."
else:
    qps_interpretation = "POOR — Student failed to learn teacher quality; re-examine pipeline."
print(f"  Interpretation: {qps_interpretation}")

print("\n── Bloom-Level Breakdown ────────────────────────────────────────────────")
bloom_levels = ["remember", "understand", "apply", "analyze", "evaluate", "create"]
print(f"  {'Bloom Level':<14} {'Teacher':>9} {'Pre-FT':>9} {'Post-FT':>9}  {'QPS %':>7}")
print("  " + "─"*52)
for bl in bloom_levels:
    t = stats_teacher.get("bloom_breakdown", {}).get(bl, {}).get("mean", "-")
    b = stats_pre_ft.get("bloom_breakdown", {}).get(bl, {}).get("mean", "-")
    a = stats_post_ft.get("bloom_breakdown", {}).get(bl, {}).get("mean", "-")
    q = qps(a, t) if isinstance(t, float) and isinstance(a, float) else "-"
    tf = f"{t:.3f}" if isinstance(t, float) else "   -  "
    bf = f"{b:.3f}" if isinstance(b, float) else "   -  "
    af = f"{a:.3f}" if isinstance(a, float) else "   -  "
    qf = f"{q:.1f}%" if isinstance(q, float) else "   -  "
    print(f"  {bl:<14} {tf:>9}  {bf:>9}  {af:>9}  {qf:>7}")

print("\n── Output Files ─────────────────────────────────────────────────────────")
print(f"  Student Pre-FT raw outputs  : {STUDENT_PRE_OUTPUT_PATH}")
print(f"  Student Post-FT raw outputs : {STUDENT_POST_OUTPUT_PATH}")
print(f"  Teacher judge scores        : {TEACHER_SCORES_PATH}")
print(f"  Student Pre-FT scores       : {STUDENT_PRE_SCORES_PATH}")
print(f"  Student Post-FT scores      : {STUDENT_POST_SCORES_PATH}")
print(f"  Final comparison JSON       : {COMPARISON_PATH}")
print(f"  Plots directory             : {PLOT_DIR}")
print()
print("  V1_radar_all_stages.png         — Spider chart: 6 dims, 3 stages")
print("  V2_grouped_bar_dimensions.png   — Grouped bars per dimension")
print("  V3_bloom_level_lines.png        — CMQS by Bloom level")
print("  V4_quality_distribution.png     — Quality label stacked bar")
print("  V5_gap_heatmap.png              — Gap heatmap Teacher vs Student")
print("  V6_cmqs_bar.png                 — Composite CMQS comparison")
print()
print("✓ Evaluation complete.")



╔════════════════════════════════════════════════════════════════╗
║      AGENTIC AI TUTOR — FINAL EVALUATION COMPARISON REPORT     ║
╚════════════════════════════════════════════════════════════════╝

  Judge model    : llama-3.3-70b-versatile
  Eval sample N  : 100
  Teacher N      : 91 valid MCQs scored
  Pre-FT N       : 85 valid MCQs scored
  Post-FT N      : 89 valid MCQs scored

── Main Results ─────────────────────────────────────────────────────────
  Dimension                             Teacher    Pre-FT   Post-FT   FT Gain    QPS %
  ──────────────────────────────────────────────────────────────────────────
  D1 — Factual Grounding                  4.758      3.788      4.045    +0.257    85.0%
  D2 — Question Clarity                   5.000      4.753      4.955    +0.202    99.1%
  D3 — Correct Answer Validity            5.000      3.929      4.472    +0.542    89.4%
  D4 — Distractor Plausibility            3.978      3.682      3.832    +0.149    96.3%
  D5 — Bloom Ali